# Requirements

## Install necessary packages

In [ ]:
"Install the necessary files"
!pip install arch
!pip install vectorbt
!pip install scikit-image
!pip install plotly==5.24.1 kaleido==0.2.1

## Import the necessary Packages

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import pickle
import copy
import re
import os
import random
import plotly.graph_objects as go
import bisect
import warnings
from collections import deque
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr,ks_2samp
#Volatility Modelling
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from skimage.filters import threshold_otsu
#Backtest
import vectorbt as vbt
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns

# Basic Tree Structure

In [ ]:
class TreeNode:
  def __init__(self, val=0, height=1, ismut=False, left=None, right=None):
    '''
    Tree structure used for the genetic algorithm (GA).
    Internal nodes are operators, and leaves are base strategies.
    '''
    self.val = val
    self.left = left
    self.right = right
    self.height = height
    self.ismut = ismut

  def __repr__(self):
    return f"TreeNode({self.val})"

# Hyperparameters to be used

## Fixed Params

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

#Global random state
global_random_state = random.Random(SEED)
global_np_random_state=np.random.RandomState(SEED)


#Yang-Zhen window length considered to calculate true volatility for a given day
yz_window = 30
#Parameters for conditional volatility
vol_params={'p':1,'q':1,'o':1,'power':2,'dist':"StudentsT"}


#Number of Technical Indicators/Strategies/Base Alphas we intend to use(the first 5 columns are OHCLV values)
num_indicators=53
#Number of operators to combine the indicators
num_bin_operators=5
num_operators=6



#Number of individuals you want to generate
num_individuals=150
num_elite=0
frac_elite=0.0#fraction of elite population you want to preserve


#Set the threshold for similarity of strategies
sim_threshold=0.8
#Rate limits for crossover
R_min_cross,R_max_cross=0.4,0.9
#Rate limits for mutation
R_min_mut,R_max_mut=0.1,0.3
#Need to define the following values each for cross-over and mutation effectively
beta1,beta2,eta=0.9,0.999,1e-3


#Backtest Params
asset_lab="SP500"
init_cash=10000 # the amount you begin with
sl_stop=0.05 # stop loss in percentage
tp_stop=0.03 # take profit in percentage
size_port=5 # the percentage of current portfolio value you want to place order
size_port_type='percent'
slippage=0.001
fees=0.001
freq='1D' #the type of dataset day,mins,hour...
#signal threshold beyond which the newly computed signal can be buy or sell
signal_threshold=0.01



# integration params
num_generations=30
stopping_generation=6
stop_threshold=0.01
ini_prev_cross,ini_prev_cross_mom,ini_curr_cross,ini_prev_cross_vel=0.7,1e-12,0.7,1e-12
ini_prev_mut,ini_prev_mut_mom,ini_curr_mut,ini_prev_mut_vel=0.3,1e-12,0.3,1e-12
num_depth=4
num_regimes,vanilla_window=2,100#no.of regimes we want to classify into
dataset_iteration=0#denotes the sub-dataset being used


#final execution params
#the indices for the training and and testing rolling window datasets along with the no. of days
#to shift the combined dataset for robust training and testing over the entire dataset
train_start,train_end,test_start,test_end,sliding_window=0,1000,1000,1500,150
fixed_train_len=train_end-train_start
fixed_test_len=test_end-test_start


#warmstart params
warmstart_percent=0.9#controlling this will control the consecutivewarmstarts
ini_warm_fac=2
curr_warmstart_percent=warmstart_percent
warmstart_tree_high=[TreeNode(i) for i in range(num_indicators)]
warmstart_tree_low=[TreeNode(i) for i in range(num_indicators)]

#avg_sharpe_dict:the avg of top 10% of the strategies for each dataset,avg_test_res:avg of the best strategy for each depth
avg_sharpe_dict,avg_test_res={},{}
#Get the depthwise relevant information in form of dictionary
dict_low,dict_high,dataset_iteration={},{},0
for d in range(2,num_depth+1):
  avg_sharpe_dict[d]=[]
  avg_test_res[d]=0.0

## Ablation Params

In [ ]:
#Hyperparameters for ablation:
#If we want to do fitness sharing
is_simulated=True

exp_del_p={}#storing the population of the best individual
price_del_p={}#the one calculated using price and particle filter

#If we want to do consecutive datasets warmstart
is_consecutive_warmstart=True
if(not is_consecutive_warmstart):
  warmstart_percent=0.0
  curr_warmstart_percent=warmstart_percent

#If we want to do fixed rates for crossover and mutation
is_fixed_rate=True#False:Adaptive rates will be used ,True:we are using fixed rates for crossover/mutation
fixed_cross_rate=1e-8
fixed_mut_rate=1e-8

#Initialize the above hyperparameters
for d in range(2,num_depth+1):
  avg_sharpe_dict[d]=[]
  avg_test_res[d]=0.0
  exp_del_p[d]={}
  price_del_p[d]={}
print(warmstart_percent,curr_warmstart_percent)

0.0 0.0


## Simulated Annealing Ablation


In [ ]:

from collections import defaultdict
#before selection the modified fitness and org_fitness
def freq_based_cdf(fitness_arr):
    """
    Input: list of (value, original_index)
    Returns:
      groups: dict value -> sorted list of original indices
      cdf:    dict value -> cumulative probability F(value)
    """
    n = len(fitness_arr)
    groups = defaultdict(set)
    for val, idx in fitness_arr:
        groups[val].add(int(idx))

    sorted_vals = sorted(groups.keys())
    counts = {v: len(groups[v]) for v in sorted_vals}

    cdf = {}
    cum = 0
    for v in sorted_vals:
        cum += counts[v]
        cdf[v] = cum / n  # frequency-based cumulative

    return groups, cdf

# Optional: map CDF back to each original index if you need per-individual CDF
def cdf_per_index(fitness_arr):
    groups, cdf = freq_based_cdf(fitness_arr)
    per_index = {}
    for val, idx in fitness_arr:
        per_index[int(idx)] = cdf[val]
    return per_index



def get_best_niche_info(org_fitness_arr):
    """
    Finds the proportion (p) and index of the best niche.
    """
    sorted_fit_arr = sorted(org_fitness_arr, key=lambda x: x[0],reverse=True)
    for rank in range(1,len(sorted_fit_arr)):
      if(sorted_fit_arr[rank][0]<sorted_fit_arr[0][0]):
        return rank/len(sorted_fit_arr),sorted_fit_arr[rank-1][1]
    return 1.0,len(sorted_fit_arr)-1


def predict_delta_p(org_fitness_arr, modified_fitness_arr, tournament_size=3):
    """
    Calculates the theoretically predicted change in population, Δp.
    """
    #Proportion (p) and index of the best niche from RAW fitness
    p, best_idx = get_best_niche_info(org_fitness_arr)
    N=len(org_fitness_arr)
    #Calculate the ECDF of the MODIFIED fitness landscape
    groups,cdfs=freq_based_cdf(modified_fitness_arr)
    if p == 1.0: # If the niche has taken over, there can be no more change
        return org_fitness_arr,groups,0.0
    sorted_vals = sorted(cdfs.keys())
    cdf_values = np.array([cdfs[v] for v in sorted_vals])

    # Calculate reproductive success w for each unique value
    w_values = (tournament_size / (N ** (tournament_size - 2))) * (cdf_values ** (tournament_size - 1))

    # Find w for the niche that contains best_idx
    w_bar = np.mean(w_values)
    w_best = w_bar
    for i, v in enumerate(sorted_vals):
        if best_idx in groups[v]:
            w_best = w_values[i]
            break


    if w_bar == 0: # Avoid division by zero
        return org_fitness_arr,groups,0.0

    # 5. Calculate \Deltap using the Price equation form
    delta_p = p * (w_best - w_bar) / w_bar
    return org_fitness_arr,groups,delta_p

# Dataset Preprocessing

In [ ]:
#Load the dataset-comment this for Local Loading
from google.colab import files
file=files.upload()

In [ ]:
# The first 5 columns of you're dataset must be C,H,L,O,V
file_path = "zscored_indicators_spy.csv"# replace with the filename you want to use
df = pd.read_csv(file_path).drop(labels=['Unnamed: 0','Date'], axis=1, errors='ignore')
display(df.head())

,Close,High,Low,Open,Volume,ADX_35,ADX_75,MACD_12_26,MACD_20_75,PLUS_DI_15,...,TSF_107,CDL3OUTSIDE,CDLHARAMI,CDLHIKKAKE,CDLRICKSHAWMAN,AD,ADOSC_48_71,ADOSC_27_73,OBV_SLOPE_22,OBV_SLOPE_113
0,1106.619995,1108.599976,1097.339966,1101.719971,1276000000,-0.476030,-1.338971,-0.766716,-1.533677,-0.191507,...,0.511115,0.0,0.0,-0.128334,-0.367423,0.618314,-1.736214,-1.849544,-0.387481,0.083194
1,1099.689941,1106.619995,1099.260010,1106.619995,1338300000,-0.483691,-1.334572,-0.687118,-1.548933,-0.359947,...,0.227180,0.0,0.0,-0.128334,-0.367423,0.078478,-1.735707,-1.836361,-0.188633,0.136416
2,1098.630005,1102.449951,1092.400024,1099.689941,1369200000,-0.460244,-1.320624,-0.629820,-1.559473,-0.583462,...,0.176114,0.0,0.0,1.950351,2.573982,0.209603,-1.726323,-1.807066,-0.108847,0.165051
3,1080.699951,1098.790039,1079.979980,1098.630005,1397400000,-0.382268,-1.290215,-0.827961,-1.711696,-0.954859,...,-0.527258,0.0,0.0,-0.146801,-0.384618,-0.391563,-1.743405,-1.826049,-0.155388,0.165871
4,1063.969971,1080.699951,1062.229980,1080.699951,1521000000,-0.233201,-1.237899,-1.201966,-1.969957,-1.263574,...,-1.175302,0.0,0.0,-0.128334,-0.384618,-0.981661,-1.782627,-1.883898,-0.202334,0.163540


In [ ]:
#Define OHLCV columns as a set for efficient lookup
start_col=5
cols_to_shift = df.columns[start_col:len(df.columns)]
df.fillna(0, inplace=True)
ohlcv_cols_set = {'Open', 'High', 'Low', 'Close','Volume'}
feature_cols = [col for col in df.columns if col not in ohlcv_cols_set]
random.shuffle(feature_cols)
#This maps original shuffled names to f1, f2, ...
rename_map = {col: f'f{i+1}' for i, col in enumerate(feature_cols)}
df.rename(columns=rename_map, inplace=True)

#Reordering the DataFrame
ohlcv_cols_list = ['Open', 'High', 'Low', 'Close','Volume']
f_cols = sorted([col for col in df.columns if re.match(r'^f\d+$', col)],
                key=lambda x: int(x[1:]))
final_col_order = ohlcv_cols_list + f_cols
df = df[final_col_order]

In [ ]:
#Preprocessing for warmstart based training data

def test_signal_generator(optim_tree,base_signals):
    final_signal=np.array([tree_signal(base_signals,optim_tree)][0])
    #Apply thresholding to classify signals
    final_signal = np.where(final_signal >signal_threshold, 1, np.where(final_signal < -signal_threshold, -1, 0))
    return final_signal


def dataset_preprocess(df,indicator_cols,start_idx,end_idx,isfirst=False,istest=False):
    """
    Preprocesses the dataset to extract base signals for training or testing.

    Args:
        df (pd.DataFrame): The input DataFrame.
        indicator_cols (list): List of column names for indicators.
        start_idx (int): Start index for slicing the DataFrame.
        end_idx (int): End index for slicing the DataFrame.
        isfirst (bool): Flag to indicate if it's the first dataset iteration for warmstart.
        istest (bool): Flag to indicate if it's for the test set.

    Returns:
        If istest: Array of base signals for the test set.
        If isfirst: Tuple of (list of base trees, NumPy array of base signals) for the initial warmstart.
        Otherwise array of base signals for the specified training range.
    """
    if istest:
       return df[indicator_cols].values.T #For testing, return signals for the entire test DataFrame
    #For training, return signals for the specified range (entire training window or regime)
    base_signals = df[indicator_cols][start_idx:end_idx].values.T
    if isfirst:
        #For the very first training window, also generate base trees
        base_trees = [TreeNode(i) for i in range(len(indicator_cols))]
        return base_trees, base_signals
    return base_signals #For subsequent training windows or regimes, just return the signals

# Volatility Modelling

## Yang-Zhen Modelling

In [ ]:
def calc_true_volatility(df,yz_window=30):
    """
    This method is used to model the true-historical volatility.
    We use the Yang-Zhen Model for this purpose.
    """
    for col in ['Open', 'High', 'Low', 'Close']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['Open', 'High', 'Low', 'Close']).reset_index(drop=True)
    open_prices = df['Open']
    high_prices = df['High']
    low_prices = df['Low']
    close_prices = df['Close']

    #Calculate returns
    r_cc = np.log(close_prices / close_prices.shift(1)).dropna()  # Close-to-Close
    r_oc = np.log(open_prices / close_prices.shift(1)).dropna()   # Open-to-Previous-Close
    r_s = (np.log(high_prices / open_prices) * np.log(high_prices / close_prices) +
           np.log(low_prices / open_prices) * np.log(low_prices / close_prices)).dropna()  # Rogers-Satchell

    r_cc = r_cc.to_numpy()
    r_oc = r_oc.to_numpy()
    r_s = r_s.to_numpy()

    d = len(r_cc)
    k = 0.34 / (1.34 + ((yz_window + 1) / (yz_window - 1)))


    #Preallocate
    yz_vol = np.full(d, np.nan)

    #Rolling calculation
    for i in range(yz_window, d):
        sigma_c = np.std(r_cc[i - yz_window:i], ddof=1)
        sigma_o = np.std(r_oc[i - yz_window:i], ddof=1)
        sigma_rs = np.mean(r_s[i - yz_window:i])
        yz_vol[i] = np.sqrt(k * sigma_o**2 + (1 - k) * sigma_c**2 + sigma_rs)
    for i in range(yz_window):
        sigma_c = np.std(r_cc[:i], ddof=1) if i > 0 else 1e-8
        sigma_o = np.std(r_oc[:i], ddof=1) if i > 0 else 1e-8
        sigma_rs = np.mean(r_s[:i])
        if(np.isnan(sigma_c)):
            sigma_c=1e-8
        if(np.isnan(sigma_o)):
            sigma_o=1e-8
        if(np.isnan(sigma_rs)):
            sigma_rs=1e-8
        yz_vol[i] = np.sqrt(k * sigma_o**2 + (1 - k) * sigma_c**2 + sigma_rs)

    return yz_vol
historical_volatility=calc_true_volatility(df,yz_window=yz_window)

## Conditional Volatility Model Selection

In [ ]:
def volatility_selector(df,method='bootstrap'):
  """
  This method is used to select the volatility model that will be employed
  over the entire dataset.We use the bootstrap method to forecast and the vol_params
  defined earlier to train the model.In order to experiment with different models we can
  change the params defined above.Based on that we select the model with lower AIC/BIC scores and RMSE value over test dataset.
  """
  returns=df['Close'].pct_change().dropna()
  returns=(returns*100)
  train_end=0.85*len(returns)
  train_returns=np.array(returns[:int(train_end)])
  test_returns=np.array(returns[int(train_end):])
  volatility_model=arch_model(train_returns,p=vol_params['p'],
                            q=vol_params['q'],
                            o=(vol_params['o'] if vol_params['o']!=-1 else 0),
                            power=(vol_params['power'] if vol_params['power']!=-1 else 2),
                            dist=(vol_params['dist'] if 'dist' in vol_params.keys() else "normal"))
  volatility_model_fit=volatility_model.fit(disp='off')
  train_volatility=volatility_model_fit.conditional_volatility
  test_volatility=volatility_model_fit.forecast(horizon=len(test_returns),method=method).variance.values[-1]
  train_historical_volatility=historical_volatility[:int(train_end)]
  test_historical_volatility=historical_volatility[int(train_end):]
  f1=(np.max(train_volatility)-np.min(train_volatility))/(np.max(train_historical_volatility)-np.min(train_historical_volatility))
  f2=(np.max(test_volatility)-np.min(test_volatility))/(np.max(test_historical_volatility)-np.min(test_historical_volatility))
  train_volatility=train_volatility*(1/f1)
  test_volatility=test_volatility*(1/f2)
  rmse = np.sqrt(np.mean((test_historical_volatility-test_volatility) ** 2))
  #print("RMSE:", rmse)
  return rmse

## Volatility Classifier

In [ ]:
def volatility_classifier_old(dataset):
  #Calculate the corresponding percentwise returns
  log_returns = dataset.pct_change().dropna()
  log_returns = np.abs(log_returns*100)

  #Fit the GARCH model
  model = arch_model(log_returns,
                     p=vol_params['p'],
                     q=vol_params['q'],
                     o=vol_params['o'],
                     power=vol_params['power'],
                     dist=vol_params['dist'])
  garch_result = model.fit(disp="off")
  predict_vol = garch_result.conditional_volatility#for train
  window = vanilla_window
  block_volatility=[np.mean(predict_vol[i:(i+window)])for i in range(0,len(predict_vol),window)]

  #Finally classify as High or Low
  vol_mean=np.mean(block_volatility)
  block_volatility=np.array([int(r>(vol_mean)) for r in block_volatility])

  #Now classify the data into high and low volatility regions and indexes
  start_high_ind,end_high_ind,start_low_ind,end_low_ind=[],[],[],[]
  high_vol_dataset,low_vol_dataset=[],[]
  #Get the datasets for different regions
  curr=dataset.iloc[:window,:]
  start_ind,end_ind=0,window
  for i,vol in enumerate(block_volatility):
    if(not i):
      continue
    if(vol!=block_volatility[i-1]):
      if(vol==1):
        start_low_ind.append(start_ind)
        end_low_ind.append(end_ind)
        low_vol_dataset.append(curr)
      else:
        start_high_ind.append(start_ind)
        end_high_ind.append(end_ind)
        high_vol_dataset.append(curr)
      start_ind=end_ind
      curr=dataset.iloc[(i-1)*window:i*window,:]
    else:
      curr=pd.concat([curr,dataset.iloc[(i-1)*window:i*window,:]])
    end_ind+=window
  if(block_volatility[-1]==1):
    start_high_ind.append(start_ind)
    end_high_ind.append(end_ind)
    high_vol_dataset.append(curr)
  else:
    start_low_ind.append(start_ind)
    end_low_ind.append(end_ind)
    low_vol_dataset.append(curr)
  return garch_result,predict_vol,high_vol_dataset,low_vol_dataset,start_high_ind,end_high_ind,start_low_ind,end_low_ind

def volatility_classifier(dataset):
    """
    Classifies a dataset into high and low volatility regimes using a GARCH model,
    Otsu's method, and returns the contiguous regime datasets and their indices.
    """
    #Calculate returns and Fit GARCH model
    returns = dataset.pct_change().dropna()
    abs_returns = np.abs(returns * 100)
    model = arch_model(abs_returns, p=vol_params['p'], q=vol_params['q'], o=vol_params['o'],
                       power=vol_params['power'], dist=vol_params['dist'])
    garch_result = model.fit(disp="off")
    predict_vol_np = garch_result.conditional_volatility
    predict_vol = np.array(predict_vol_np)

    #Calculate the OPTIMAL threshold using Otsu's method
    otsu_threshold = threshold_otsu(predict_vol)

    #Classify the BLOCKS using the new threshold
    block_volatility = [np.mean(predict_vol[i:(i + vanilla_window)])
                        for i in range(0, len(predict_vol), vanilla_window)]
    block_classification = np.array([1 if r > otsu_threshold else 0 for r in block_volatility])

    #Identify regime start and end indices based on block classifications
    start_high_ind, end_high_ind = [], []
    start_low_ind, end_low_ind = [], []

    if len(block_classification) == 0:
        return garch_result, predict_vol, [], [], [], [], [], []

    current_regime = block_classification[0]
    regime_start_block = 0

    #Loop through the block classifications to find where regimes change
    for i in range(1, len(block_classification)):
        if block_classification[i] != current_regime:
            end_index = i * vanilla_window
            start_index = regime_start_block * vanilla_window

            if current_regime == 1: # High volatility regime ended
                start_high_ind.append(start_index)
                end_high_ind.append(end_index)
            else: # Low volatility regime ended
                start_low_ind.append(start_index)
                end_low_ind.append(end_index)

            current_regime = block_classification[i]
            regime_start_block = i

    start_index = regime_start_block * vanilla_window
    end_index = min(len(block_classification) * vanilla_window, len(predict_vol))

    if current_regime == 1:
        start_high_ind.append(start_index)
        end_high_ind.append(end_index)
    else:
        start_low_ind.append(start_index)
        end_low_ind.append(end_index)

    #Initialize empty lists to hold the DataFrame chunks
    high_vol_dataset = []
    low_vol_dataset = []

    #Populate the high volatility dataset list
    for start, end in zip(start_high_ind, end_high_ind):
        high_vol_dataset.append(dataset.iloc[start:end])

    #Populate the low volatility dataset list
    for start, end in zip(start_low_ind, end_low_ind):
        low_vol_dataset.append(dataset.iloc[start:end])
    print(f"The number of high ds are {len(high_vol_dataset)} and that of low are {len(low_vol_dataset)}")
    return garch_result, predict_vol, high_vol_dataset, low_vol_dataset, start_high_ind, end_high_ind, start_low_ind, end_low_ind

# Tree Structure

## Tree Definition

In [ ]:
def tree_signal(base_signals, node):
    """
    Recursively evaluates a signal tree using base signals and operators.

    Args:
        base_signals (list of np.array): Base strategy signals.
        node (TreeNode): Root of the expression tree.

    Returns:
        np.array: Final signal with dtype np.int8.
    """
    if node is None:
        return np.zeros(len(base_signals[0]), dtype=np.float16)
    if node.left is None and node.right is None:
        node.val=min(node.val,len(base_signals)-1)
        return base_signals[node.val].astype(np.float16)
    result = np.zeros(len(base_signals[0]), dtype=np.float16)
    # Evaluate left and/or right children as needed
    if node.val in {5,6,7,8,9,10}:  # Unary operators
        child_signal = tree_signal(base_signals, node.left).astype(np.float16)

        if node.val == 5:  # abs
            result = np.abs(child_signal)
        elif node.val == 6:  # cos
            result = np.cos(child_signal)
        elif node.val == 7: # sin
            result = np.sin(child_signal)
        elif node.val == 8:  # tan
            result = np.tan(child_signal)
        elif node.val == 9:  # exp
            result = np.exp(child_signal)
        elif node.val == 10:  # log (safe)
            safe_signal = np.where(child_signal > 0, child_signal, 1e-6)
            result = np.log(safe_signal)

    else:  # Binary operators
        left = tree_signal(base_signals, node.left).astype(np.float16)
        right = tree_signal(base_signals, node.right).astype(np.float16)

        if node.val == 0:  # sum
            result = left + right
        elif node.val == 1:  # subtract
            result = left - right
        elif node.val == 2:  # multiply
            result = left * right
        elif node.val == 3:  # max
            result = np.maximum(left, right)
        elif node.val == 4:  # min
            result = np.minimum(left, right)
    return result.astype(np.float16)


## Tree Utils

In [ ]:
def get_height(root, visited=None):
    if root is None:
        return 0
    if visited is None:
        visited = set()
    if id(root) in visited:  #cycle detected
        return 0
    visited.add(id(root))
    return 1 + max(get_height(root.left, visited), get_height(root.right, visited))



def bfs(root, depth,random_state=global_random_state):
  """
  This function performs a breadth-first search(BFS) to find a node at a given depth.(used in crossover/mutation)
  Args:
      root (TreeNode): The root node of the tree.
      depth (int): The depth to search for.
  Returns:
      TreeNode: The node at the given depth, or None if not found.
  """
  if random_state is None:
        random_state = random  #Use global random if no local random_state is provided
  if depth == 0 or not root:
    return root

  queue = deque([root])

  for _ in range(depth):  #Iterate through levels
    if not queue:  #If we run out of nodes, return None
      return None
    for _ in range(len(queue)):  #Process each level
      node = queue.popleft()
      if node.left:
        queue.append(node.left)
      if node.right:
        queue.append(node.right)

  return random_state.choice(queue) if queue else None


def unary_create_tree(tree,op_node,val):
    """
    This function increases the depth of the tree using binary operator
    Args:
        tree(TreeNode) : The strategy tree
        op_node(TreeNode): The unary operator to add on the tree
        val(int): The value of the op_node
    Returns:
        TreeNode :The new tree after addition of unary operator
    """
    op_node.val = val
    op_node.left = tree
    op_node.height = get_height(op_node)
    return op_node

def binary_create_tree(tree1,tree2,op_node,val):
    """
    This function increases the depth of the tree using unary operator
    Args:
        tree1(TreeNode) : The first strategy tree
        tree2(TreeNode) : The second strategy tree
        op_node(TreeNode): The binary operator to add on the tree
        val(int): The value of the op_node
    Returns:
        TreeNode :The new tree after addition of binary operator
    """
    op_node.val = val
    op_node.left = tree1
    op_node.right = tree2
    op_node.height = get_height(op_node)
    return op_node

def add_depth_binary(base_pop,n,random_state=global_random_state):
    """
    This function increases the depth of the tree using binary operator
    for the list of input strategies array
    Args:
        base_pop(List[TreeNode]) : Array of strategy trees
        n(int):No of binary  trees to be created
    Returns:
        List[TreeNode] :The new tree(strategy) population after combining
                        them with binary operator
    """
    if random_state is None:
        random_state = random
    count = len(base_pop)
    total_pairs = (int)(0.5*count*(count-1))
    if(total_pairs<n):
      n=total_pairs
    chosen_indices = random_state.sample(range(1,total_pairs),n-1)
    base_pop_new = []
    s,ps,c= 0,[],count-1
    while c > 0:
      s+=c
      ps.append(s)
      c-=1

    for i in chosen_indices:
        left_id = bisect.bisect_right(ps, i)
        rr = i - ps[max(left_id-1,0)]
        right_id = left_id + rr
        root = TreeNode()
        root = binary_create_tree(base_pop[left_id], base_pop[right_id], root,global_np_random_state.randint(0,num_bin_operators))
        base_pop_new.append(root)

    return base_pop_new

def add_depth_unary(base_pop,n,random_state=global_random_state):
  """
  This function increases the depth of the tree using unary operator
  for the list of input strategies array
  Args:
      base_pop(List[TreeNode]) : Array of strategy trees
      n(int):No of unary  trees to be created
  Returns:
      List[TreeNode] :The new tree(strategy) population after combining
                      them with unary operator
  """
  if random_state is None:
        random_state = random
  chosen_indices = random_state.sample(range(0,len(base_pop)),n)
  base_pop_new = []
  for i in chosen_indices:
        root = TreeNode()
        root = unary_create_tree(base_pop[i], root,global_np_random_state.randint(num_bin_operators,num_operators))
        base_pop_new.append(root)
  return base_pop_new

## Similarity Structure of Trees

In [ ]:
def check_same(tree1, tree2):
    if tree1 is None and tree2 is None:
        return True
    if tree1 is None or tree2 is None:
        return False
    if tree1.val != tree2.val:
        return False

    #Check for commutative operators
    if tree1.val in [0,2,3,4]:
        return ((check_same(tree1.left, tree2.left) and check_same(tree1.right, tree2.right)) or
                (check_same(tree1.left, tree2.right) and check_same(tree1.right, tree2.left)))

    #All other operators are compared normally
    return check_same(tree1.left, tree2.left) and check_same(tree1.right, tree2.right)


In [ ]:
COMMUTATIVE_OPS = {0, 2, 3, 4}
ASSOCIATIVE_OPS = {0, 2}

def flatten_associative(node, target_op):
    if node is None:
        return []
    if node.val == target_op:
        return flatten_associative(node.left, target_op) + flatten_associative(node.right, target_op)
    else:
        return [node]

def canonical_tree_fingerprint(node):
    if node is None:
        return "N"

    # Leaf node
    if node.left is None and node.right is None:
        return f"L{node.val}"

    if node.val in ASSOCIATIVE_OPS:
        flat_nodes = flatten_associative(node, node.val)
        flat_fps = sorted(canonical_tree_fingerprint(n) for n in flat_nodes)
        return f"A{node.val}({','.join(flat_fps)})"

    elif node.val in COMMUTATIVE_OPS:
        left_fp = canonical_tree_fingerprint(node.left)
        right_fp = canonical_tree_fingerprint(node.right)
        children = sorted([left_fp, right_fp])
        return f"C{node.val}({children[0]},{children[1]})"

    else:
        left_fp = canonical_tree_fingerprint(node.left)
        right_fp = canonical_tree_fingerprint(node.right)
        return f"B{node.val}({left_fp},{right_fp})"

#Main Matching and Tracking Code
def track_duplicate_trees(trac_tree_dic_high_,warm_strategy_pop_):
    #Precompute fingerprint set of the reference population
    warm_fingerprints = {
        canonical_tree_fingerprint(tree) for tree in warm_strategy_pop_
    }

    #Collect matching trees
    dum_trac_tree_dic = [
        tree for tree in trac_tree_dic_high_
        if canonical_tree_fingerprint(tree) in warm_fingerprints
    ]
    return len(dum_trac_tree_dic)


# Backtest Function

In [ ]:
class BuyAndHoldBacktest:
    def __init__(self, dataset):
        if 'Close' not in dataset.columns:
            raise ValueError("Dataset must contain a 'Close' column.")

        self.df = dataset.copy()
        asset_label = asset_lab

        # Create entry and exit signals
        entries = pd.Series(False, index=self.df.index)
        exits = pd.Series(False, index=self.df.index)
        entries.iloc[0] = True  #Enter on the first day
        exits.iloc[-1] = True   #Exit on the last day

        #Create a portfolio with the buy-and-hold strategy
        self.portfolio = vbt.Portfolio.from_signals(
            close=self.df['Close'],
            entries=entries,
            exits=exits,
            init_cash=init_cash,
            fees=fees,
            slippage=slippage,
            freq=freq
        )

    def get_portfolio(self):
        return self.portfolio


In [ ]:
class SingleVectorBacktest:
    def __init__(self,dataset,terminal_signal):
        """
        Perform backtesting on a single trading signal vector.

        Args:
            dataset (pd.DataFrame): DataFrame with at least a 'Close' column.
            terminal_signal (pd.Series or np.array): 1D array of signals (1=buy, -1=sell, 0=hold).
            asset_lab (str): The label for the asset being traded.
            ... other vectorbt parameters
        """
        self.df = dataset.copy()

        #Shift the signal by 1 to prevent lookahead bias.
        shifted_signal = pd.Series(terminal_signal, index=self.df.index).shift(1).fillna(0)

        #Set the last position to -1 (exit) for all strategies
        shifted_signal.iloc[-1] = -1

        entries = shifted_signal == 1
        exits = shifted_signal == -1
        price = self.df['Close']

        self.portfolio = vbt.Portfolio.from_signals(
            price,
            entries,
            exits,
            init_cash=init_cash,
            slippage=slippage,
            fees=fees,
            freq=freq
        )

    def port_ret(self):
        return self.portfolio

    def fitness(self, type="information_ratio"):
        #Calculate benchmark returns (Buy and Hold) directly from the price data
        benchmark_rets = self.df['Close'].vbt.to_returns()
        metrics = self.port_ret()

        if type == "sharpe":
            return metrics.sharpe_ratio()
        elif type == "information_ratio":
            return 100 * metrics.information_ratio(benchmark_rets=benchmark_rets)
        elif type == "max_drawdown":
            return -metrics.max_drawdown()
        elif type == "calmar_ratio":
            return metrics.calmar_ratio()
        elif type == "sortino_ratio":
            return metrics.sortino_ratio()
        elif type == "omega_ratio":
            return metrics.omega_ratio()
        else:
            pass




In [ ]:
class VectorBacktest:
  def __init__(self,dataset,terminal_signals,fusion=False):
    """
    Perform backtesting on trading strategies using decision trees for signal generation.
    The backtest is conducted using the VectorBT library for portfolio simulation.

    Attributes:
        strategy_tree (TreeNode): The decision tree representing the strategy.
        dataset (pd.DataFrame): Historical price data for backtesting.
        terminal_signals (list[list[int]]): Predefined base signals for each time point.
        fusion (bool): Whether to apply Kalman smoothing to the generated signals.
    """
    self.df = dataset.copy()
    self.test_signal = np.array(terminal_signals)
    if (self.test_signal.shape[1] != len(self.df)):
      print(self.test_signal.shape[1],len(self.df))
    assert self.test_signal.shape[1] == len(self.df), "Signals must match dataset length"
    shifted_signals = np.roll(self.test_signal, shift=1, axis=1)

    #Set the last position to -1 (exit) for all strategies
    shifted_signals[:, -1] = -1
    shifted_signals[:,0] = 0
    signal_df = pd.DataFrame(
            shifted_signals.T,
            index=self.df.index,
            columns=[f'sig{i+1}' for i in range(self.test_signal.shape[0])]
        )
    entries_df = signal_df == 1
    exits_df   = signal_df == -1
    asset_label = asset_lab
    entries = entries_df.vbt.stack_index(pd.Index([asset_label] * entries_df.shape[1], name="asset"))
    exits   = exits_df.vbt.stack_index(pd.Index([asset_label] * exits_df.shape[1], name="asset"))

    price_df = self.df[['Close']].copy()
    price_df.columns = [asset_label]
    price_df.columns.name = 'asset'

    self.portfolio = vbt.Portfolio.from_signals(
            price_df,
            entries,
            exits,
            init_cash=init_cash,
            slippage=slippage,
            fees=fees,
            freq=freq
        )
  def port_ret(self):
        # Return the portfolio object.
        return self.portfolio
  def fitness(self,type="sharpe"):
      backtest=BuyAndHoldBacktest(self.df[['Close']].copy())
      metrics=self.port_ret()
      if(type=="sharpe"):
        return metrics.sharpe_ratio()
      elif(type=="information_ratio"):
        return 100*metrics.information_ratio(benchmark_rets=backtest.get_portfolio().returns())
      elif(type=="max_drawdown"):
        return -metrics.max_drawdown()
      elif(type=="calmar_ratio"):
        return metrics.calmar_ratio()
      elif(type=="sortino_ratio"):
        return metrics.sortino_ratio()
      elif(type=="omega_ratio"):
        return metrics.omega_ratio()
      else:#here I encourage to try out combined fitness functions
        pass

# Genetic Programming Architecture

## Crossover

In [ ]:
def clone_tree(node):
    """Recursively clone a tree without parent cycles."""
    if node is None:
        return None
    new_node = TreeNode(node.val)
    new_node.left = clone_tree(node.left)
    new_node.right = clone_tree(node.right)
    return new_node
def unary_rootswap(root_a, root_b):
  """
  Function to swap subtree when root node of one subtree is unary operator and while the other one is binary.
  Args:
    root_a: Unary operator rooted subtree
    root_b:Binary operator rooted subtree
  """
  if root_a.left and root_b.left:
    left_subtree= root_a.left
    r = global_np_random_state.randint(0, 2)
    if r:
      root_a.left, root_b.right = root_b.right, left_subtree
    else:
      root_a.left, root_b.left = root_b.left, left_subtree


def binary_rootswap(node_a, node_b):
  """
  Function to swap subtrees when the root node of both are binary operators.
  Args:
    node_a: Binary operator rooted subtree
    node_b:Binary operator rooted subtree
  """
  node_a.val, node_b.val = node_b.val, node_a.val
  node_a.left, node_b.left = node_b.left, node_a.left
  node_a.right, node_b.right = node_b.right, node_a.right

def crossover(tree1, tree2):
  """
  Perform crossover between two strategy trees at random depths.
  Args:
    tree1 (TreeNode): First strategy tree.
    tree2 (TreeNode): Second strategy tree.
  Returns:
     tuple: Two child trees resulting from the crossover.
  """
  child1, child2 = clone_tree(tree1), clone_tree(tree2)

  height1, height2 = get_height(child1), get_height(child2)
  if (height1<= 1 or height2<=1):
    return child1,child2

  depth1 = global_np_random_state.randint(0, height1 - 1)
  depth2 = global_np_random_state.randint(0, height2 - 1)
  root1, root2 = bfs(child1, depth1), bfs(child2, depth2)

  if root1 and root2:
    #Both nodes are unary operators
    if root1.val >= num_bin_operators and root2.val >= num_bin_operators:
      root1.left, root2.left = root2.left, root1.left

    #Only one is unary
    elif root1.val >= num_bin_operators:
      unary_rootswap(root1, root2)
    elif root2.val >= num_bin_operators:
      unary_rootswap(root2, root1)

    #Perform standard binary swap
    else:
      swap_choice = global_random_state.choice(['left_left', 'right_right', 'left_right', 'right_left'])
      if swap_choice == 'left_left' and root1.left and root2.left:
        binary_rootswap(root1.left, root2.left)
      elif swap_choice == 'left_right' and root1.left and root2.right:
        binary_rootswap(root1.left, root2.right)
      elif swap_choice == 'right_left' and root1.right and root2.left:
        binary_rootswap(root1.right, root2.left)
      elif swap_choice == 'right_right' and root1.right and root2.right:
        binary_rootswap(root1.right, root2.right)

  return child1, child2

## Mutation & Selection

In [ ]:
def mutation(root,num_base=num_indicators):
  """
  Perform (point)mutation on a strategy tree by modifying a random node's value.
  Args:
    root (TreeNode): Root of the strategy tree to mutate.
    num_base(int) : Number of base signals
  Returns:
    The tree is modified in-place.
  """
  if root is None:
    return root
  root = clone_tree(root)
  tree_height = get_height(root)
  random_depth = global_np_random_state.randint(0, tree_height)
  node = bfs(root, random_depth)
  node_height = get_height(node)
  if node is None:
    return root

  def set_value(beg,end):
    new_val = global_np_random_state.randint(beg,end)
    while new_val == node.val:  #Ensure mutation actually changes the value
      new_val = global_np_random_state.randint(beg,end)
    node.val = new_val
  if node_height == 1:
    set_value(0,num_base)

  else:
    if node.val >= num_bin_operators:
      if num_operators-num_bin_operators>1:
        set_value(num_bin_operators,num_operators)
      else:
        return root
    else:
      set_value(0,num_bin_operators)
  return root

def tournament_selection(fitness_arr, k = 3):
    """
    Perform tournament selection to choose a parent based on fitness.
    Args:
        fitness_arr (list): List of fitness scores for individuals.
        k (int, optional): Number of individuals to select for the tournament (default: 3).
    Returns:
        int: Index of the selected individual.
    """
    tournament = global_random_state.sample(list(enumerate(fitness_arr)), k)
    winner = max(tournament, key=lambda x: float(x[1][0]))
    id = winner[1][1]
    return id

## Adaptive Rate(ADAM Based)

In [ ]:
def _calculate_elite_mean_fitness(fitness_array, elite_perc=10.0):
    """Calculates the mean fitness of the top elite_perc percent of individuals."""
    if not fitness_array:
        return 0.0
    try:
        fit_values = np.array([t[0] if isinstance(t, tuple) else t for t in fitness_array], dtype=float)
        fit_values = fit_values[np.isfinite(fit_values)]
        if fit_values.size == 0:
            return 0.0
    except (TypeError, ValueError) as e:
        warnings.warn(f"Could not process fitness array: {fitness_array}. Error: {e}. Returning 0.0")
        return 0.0

    population_size = len(fit_values)
    if population_size == 0:
        return 0.0

    #Ensure elite_perc is within (0, 100)
    elite_perc = max(0.0, min(100.0, elite_perc))
    if elite_perc == 0.0:
        return 0.0
    num_elite = max(1, int(np.floor(population_size * elite_perc / 100.0)))
    num_elite = min(num_elite, population_size)

    sorted_fitness = np.sort(fit_values)[::-1]
    top_elite_fitness = sorted_fitness[:num_elite]

    return np.mean(top_elite_fitness)


def adam_rate_controller(
    prev_rate: float,
    curr_rate: float,
    prev_momentum: float,
    prev_velocity: float,
    prev_fitness_arr: list,
    curr_fitness_arr: list,
    beta1: float,
    beta2:float,
    eta: float,
    rate_bounds: tuple[float, float],
    curr_gen=1,
    epsilon: float = 1e-8,
    grad_clip_range: tuple[float, float] | None = (-1e6, 1e6),
    elite_perc: float = 10.0,
    add_noise: bool = True,
    noise_scale: float = 0.005,
    random_state: np.random.RandomState | None = None
) -> tuple[float, float,float,float]:
    """
    Dynamically adjusts a GA operator rate (mutation/crossover) based on fitness gradients
    using momentum.

    Args:
        prev_rate (float): Operator rate used in the previous generation.
        curr_rate (float): Operator rate used in the current generation.
        prev_momentum (float): Momentum value from the previous generation.
        prev_velocity (float): Velocity value from the previous generation.
        prev_fitness_arr (list): Fitness values from the previous generation's population.
        curr_fitness_arr (list): Fitness values from the current generation's population.
        beta1 (float): Momentum factor (e.g., 0.9). Controls smoothing.
        beta2 (float): Momentum factor for velocity (e.g., 0.999).
        eta (float): Learning rate for rate adjustment.(0.001)
        rate_bounds (tuple[float, float]): Min and Max allowed values for the rate (e.g., (0.5, 0.8)).
        curr_gen (int): Current generation number.
        epsilon (float, optional): Small value to prevent division by zero in gradient denominator. Defaults to 1e-8.
        grad_clip_range (tuple[float, float] | None, optional): Min/Max values to clip the raw gradient to.
            Set to None to disable gradient clipping. Defaults to (-10.0, 10.0).
        elite_perc (float, optional): Percentage of top individuals to consider for fitness change. Defaults to 10.0.
        add_noise (bool, optional): Whether to add small uniform noise to the updated rate. Defaults to True.
        noise_scale (float, optional): The amplitude of the uniform noise (+/- this value). Defaults to 0.005.
        random_state (np.random.RandomState | None, optional): NumPy random state generator for reproducible noise.
            If None, uses np.random. Defaults to None.

    Returns:
        tuple[float, float]: (next_rate, current_momentum) - The calculated rate for the next generation
                             and the momentum value calculated in this generation.
    """

    #Calculate change in elite fitness
    prev_top_mean = _calculate_elite_mean_fitness(prev_fitness_arr, elite_perc)
    curr_top_mean = _calculate_elite_mean_fitness(curr_fitness_arr, elite_perc)
    delta_fitness = (curr_top_mean - prev_top_mean)*10
    delta_rate_raw = curr_rate - prev_rate
    delta_rate_clipped = np.sign(delta_rate_raw) * max(abs(delta_rate_raw), epsilon)

    #Estimate Gradient
    if delta_rate_clipped == 0:
         avg_grad = 0.0
         warnings.warn("Delta rate was zero even after epsilon flooring.")
    else:
        avg_grad = delta_fitness / delta_rate_clipped

    if grad_clip_range is not None:
        avg_grad = np.clip(avg_grad, grad_clip_range[0], grad_clip_range[1])

    #Update Momentum
    curr_momentum = beta1 * prev_momentum + (1 - beta1) * avg_grad
    curr_velocity = beta2 * prev_velocity + (1 - beta2) * avg_grad**2
    curr_momentum = curr_momentum / (1 - beta1**curr_gen)
    curr_velocity = curr_velocity / (1 - beta2**curr_gen)

    next_rate = curr_rate + eta * (curr_momentum/(curr_velocity**0.5+epsilon))
    R_min, R_max = rate_bounds
    next_rate = np.clip(next_rate, R_min, R_max)
    noise = np.random.uniform(-0.00001,0.00001)
    next_rate = np.clip(next_rate + noise, R_min, R_max)
    return next_rate,curr_momentum,curr_velocity,curr_top_mean

## Similarity Scores of Strategies

In [ ]:
def calculate_similarity_matrix_np(data_matrix):
    """
    Calculates the Pearson similarity (correlation) matrix for a 2D NumPy array.

    Assumes rows are variables (e.g., PnL arrays) and columns are observations (e.g., time points).

    Args:
        data_matrix: A 2D NumPy array where each row is a data series (e.g., shape (num_individual, num_days)).

    Returns:
        A 2D NumPy array representing the pairwise Pearson correlation matrix
        (e.g., shape (num_individual,num_individual)). Returns NaN for correlations involving
        constant rows (zero standard deviation) or rows with insufficient data points.
    """
    if not isinstance(data_matrix, np.ndarray):
        raise TypeError("Input must be a NumPy array.")
    if data_matrix.ndim != 2:
        raise ValueError(f"Input must be a 2D array, but got {data_matrix.ndim} dimensions.")

    num_arrays = data_matrix.shape[0]
    num_samples = data_matrix.shape[1]

    if num_arrays == 0:
        print("Warning: Input array has 0 rows. Returning empty matrix.")
        return np.array([])

    if num_arrays == 1:
        print("Warning: Input array has only 1 row. Correlation with itself is 1.")
        return np.array([[1.0]])

    if num_samples < 2:
        print(f"Warning: Number of samples ({num_samples}) is less than 2. Cannot calculate correlation.")
        return np.full((num_arrays, num_arrays), np.nan)
    #Calculate Correlation Matrix
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        correlation_matrix = np.corrcoef(data_matrix.astype(float, copy=False))
    if correlation_matrix.shape == ():
         return np.full((num_arrays, num_arrays), np.nan)
    if correlation_matrix.shape != (num_arrays, num_arrays):
         print(f"Warning: Unexpected output shape {correlation_matrix.shape}. Expected ({num_arrays}, {num_arrays}). Returning NaN matrix.")
         return np.full((num_arrays, num_arrays), np.nan)

    return correlation_matrix


def analyze_similarity(similarity_matrix, threshold=sim_threshold):
    """
    Analyzes a similarity matrix: replaces NaNs with 0 and counts items
    above a threshold for each row (excluding self-similarity).

    Args:
        similarity_matrix: The 2D NumPy array correlation matrix (potentially containing NaNs).
        threshold: The float correlation threshold for counting neighbors.

    Returns:
        A tuple containing:
        cleaned_matrix: The similarity matrix with NaNs replaced by 0.
        counts_above_threshold: A 1D NumPy array where counts_above_threshold[i]
                                  is the number of items j (j!=i) such that
                                  cleaned_matrix[i, j] > threshold.
    """
    if not isinstance(similarity_matrix, np.ndarray) or similarity_matrix.ndim != 2:
         raise ValueError("Input similarity_matrix must be a 2D NumPy array.")
    if similarity_matrix.shape[0] != similarity_matrix.shape[1]:
         raise ValueError("Input similarity_matrix must be square.")

    cleaned_matrix = np.nan_to_num(similarity_matrix, nan=0.0)
    num_items = cleaned_matrix.shape[0]
    if num_items == 0:
        return cleaned_matrix, np.array([], dtype=int)

    #Create boolean matrix where condition (value > threshold) is met
    above_threshold_matrix = cleaned_matrix > threshold
    np.fill_diagonal(above_threshold_matrix, False)
    counts_above_threshold = np.sum(above_threshold_matrix, axis=1, dtype=int)

    return cleaned_matrix, counts_above_threshold

## Next Generation Module

In [ ]:
def fitness_suppresser(pnl_list, fitness_list, curr_gen, tot_gen):
    multiplicative_factor = np.exp(-(curr_gen)/tot_gen)
    pnl_list = np.array(pnl_list)
    initial_similarity_matrix = calculate_similarity_matrix_np(pnl_list)
    _, counts = analyze_similarity(initial_similarity_matrix)

    #Create a new list to store updated fitness values
    updated_fitness_list = []

    for idx, (fit_val, index) in enumerate(fitness_list):
        cnt = counts[index]
        if (((1.0 + cnt) * multiplicative_factor)>1):
            fit_val = fit_val/ ((1.0 + cnt) * multiplicative_factor)
        updated_fitness_list.append((fit_val, index))

    return updated_fitness_list

def tree_fitness_calc(dataset, base_signals, tree_pop, return_port=False):
    tree_signal_arr = []
    for tree in tree_pop:
        final_signal = np.array([tree_signal(base_signals, tree)][0])
        #Apply thresholding to classify signals
        final_signal = np.where(final_signal >signal_threshold, 1, np.where(final_signal < -signal_threshold, -1, 0))
        tree_signal_arr.append(final_signal)

    tree_obj = VectorBacktest(dataset, tree_signal_arr)
    if return_port:
        return tree_obj, tree_obj.port_ret()

    tree_fitness = tree_obj.fitness()
    tree_fitness = [0 if (x > 200) else max(x,-200.0) for x in tree_fitness]
    return tree_fitness

def update_rates(curr_gen, prev_fitness, curr_fitness, depth,ishigh,update_mutation=True, **kwargs):
    if update_mutation:
        tag = "mut"
        R_min, R_max, curr_rate = R_min_mut, R_max_mut, kwargs["curr_mut_rate"]

    else:
        tag = "cross"
        R_min, R_max, curr_rate = R_min_cross, R_max_cross, kwargs["curr_cross_rate"]

    curr_ad, prev_mom, prev_vel,elite_mean = adam_rate_controller(kwargs[f"prev_{tag}_rate"],kwargs[f"curr_{tag}_rate"],
                                                                  kwargs[f"prev_{tag}_mom"],kwargs[f"prev_{tag}_vel"],
                                                                  prev_fitness,curr_fitness,beta1,beta2,eta,
                                                                   (R_min, R_max),curr_gen)

    kwargs[f"prev_{tag}_rate"] = curr_rate
    kwargs[f"curr_{tag}_rate"] = curr_ad
    kwargs[f"prev_{tag}_mom"] = prev_mom
    kwargs[f"prev_{tag}_vel"] = prev_vel
    return kwargs

def simulated_next_generation(base_signals,tree_pop,train_dataset,curr_gen,
                              tot_gen,fitness_arr,#before selection-modified fitness
                              depth,ishigh,**kwargs):
    """
    Evolve the next generation of strategies using simulated annealing based genetic algorithm.
    Args:
        base_signals: Base signals for the tree
        tree_pop: The current strategy tree population
        train_dataset: The dataset being used
        curr_gen: the current generation
        tot_gen: total no. of generations
        fitness_arr: The corresponding fitness of the tree population
        depth: Maximum tree depth
        **kwargs: Additional parameters including adaptive rates
    Returns:
        tree_pop_new: new strategy trees
        avg_new_fitness_pop: average fitness of the top 10 in new population
        fitness_new_arr: new fitness array for the corresponding strategies
        fitness_new_pnl_arr: new pnl array for the corresponding strategies
        **kwargs: updated parameters for adaptive rates
    """
    size = len(fitness_arr)
    if(is_fixed_rate):
      curr_cross_rate = fixed_cross_rate
      curr_mut_rate = fixed_mut_rate
    else:
      curr_cross_rate = kwargs["curr_cross_rate"]
      curr_mut_rate = kwargs["curr_mut_rate"]

    tree_pop_new = []
    #Calculating raw fitness of previous population
    adam_prev_fitness = tree_fitness_calc(train_dataset, base_signals, tree_pop)

    old_par_fitness_arr=[x[0] for x in fitness_arr]
    sorted_fitness_arr = sorted(fitness_arr, key=lambda x: x[0], reverse=True)
    pre_selection_raw=[(x, ind) for ind, x in enumerate(adam_prev_fitness)]
    price_val = predict_delta_p(pre_selection_raw,fitness_arr)
    print(f"Dataset iteration {dataset_iteration}")
    if not ishigh:
      if dataset_iteration not in price_del_p[depth]:
          price_del_p[depth][dataset_iteration] = []
      if dataset_iteration not in exp_del_p[depth]:
          exp_del_p[depth][dataset_iteration] = []

      price_del_p[depth][dataset_iteration].append(price_val)
      exp_del_p[depth][dataset_iteration].append(get_best_niche_info(pre_selection_raw)[0])

    sel_par_fitness_mapper=[]
    for i in range(num_elite):
        id = sorted_fitness_arr[i][1]
        tree_pop_new.append(tree_pop[id])
        sel_par_fitness_mapper.append(fitness_arr[id][0])

    #Batching the population for faster inference
    child_tree_arr = []
    parent_tree_arr = []
    parent_tree_fitness_arr = []
    if np.isnan(curr_cross_rate):
        curr_cross_rate = R_max_cross

    #Create new individuals through crossover
    for i in range(size// 2):
        id1 = tournament_selection(fitness_arr)
        id2 = tournament_selection(fitness_arr)
        parent1 = tree_pop[id1]
        parent2 = tree_pop[id2]
        sel_par_fitness_mapper.append(fitness_arr[id1][0])
        sel_par_fitness_mapper.append(fitness_arr[id2][0])

        if random.random() < curr_cross_rate:
            child1, child2 = crossover(parent1, parent2)

            child_tree_arr.append(child1)
            child_tree_arr.append(child2)
            parent_tree_arr.append(parent1)
            parent_tree_arr.append(parent2)
            parent_tree_fitness_arr.append(adam_prev_fitness[id1])
            parent_tree_fitness_arr.append(adam_prev_fitness[id2])
        else:
            tree_pop_new.append(parent1)
            tree_pop_new.append(parent2)
    if child_tree_arr:
        child_sharpe = tree_fitness_calc(train_dataset, base_signals, child_tree_arr)

        for i in range(0, len(child_tree_arr), 2):
            if i+1 < len(child_tree_arr):
                if max(child_sharpe[i], child_sharpe[i+1]) > min(parent_tree_fitness_arr[i], parent_tree_fitness_arr[i+1]):
                    tree_pop_new.append(child_tree_arr[i])
                    tree_pop_new.append(child_tree_arr[i+1])
                else:
                    tree_pop_new.append(parent_tree_arr[i])
                    tree_pop_new.append(parent_tree_arr[i+1])

    #Mutate selected individuals
    if np.isnan(curr_mut_rate):
        curr_mut_rate = R_max_mut

    num_mut = int(len(tree_pop_new)*curr_mut_rate)
    if num_mut > 0:
        n_mut_arr = global_np_random_state.randint(0, len(tree_pop_new),size=num_mut)  # adaptive mutation
        for i in n_mut_arr:
            tree_pop_new[i] = mutation(tree_pop_new[i])

    #Evaluate new population fitness and penalize similarity
    fitness_new_arr, fitness_new_pnl_arr = [], []
    obj, metrics = tree_fitness_calc(train_dataset, base_signals, tree_pop_new, return_port=True)
    signal_names = metrics.value().columns

    for signal_name in signal_names:
        val = metrics.value()[signal_name].values
        fitness_new_pnl_arr.append(val)

    fitness_new_arr = [(0.0 if (x > 200 or x < -200) else x, ind) for ind, x in enumerate(obj.fitness())]
    adam_curr_fitness = [t[0] for t in fitness_new_arr]
    sorted_fit_gen_new = np.array([t[0] for t in sorted(fitness_new_arr, key=lambda x: x[0], reverse=True)])
    print(f"Current Generation population size and fitness score of elite population: {len(fitness_new_arr)}, {np.mean(sorted_fit_gen_new[:10])}")
    if is_simulated:
        fitness_new_arr = fitness_suppresser(fitness_new_pnl_arr, fitness_new_arr, curr_gen, tot_gen)

    avg_new_fitness_pop = np.mean(sorted_fit_gen_new[:10])
    if is_fixed_rate==False:
      kwargs = update_rates(curr_gen, adam_prev_fitness, adam_curr_fitness, depth,ishigh,update_mutation=False, **kwargs)
      kwargs = update_rates(curr_gen, adam_prev_fitness, adam_curr_fitness, depth,ishigh,update_mutation=True, **kwargs)
    return tree_pop_new, avg_new_fitness_pop, fitness_new_arr, fitness_new_pnl_arr, kwargs

# Warmstart

## Version 1

In [ ]:
def tree_creation_begin(base_pop):
    '''
    Generates the unary and binary rooted trees for the warmstart function
    '''
    count = len(base_pop)
    num_unary = global_np_random_state.randint(0,count)
    num_binary = int(num_individuals*ini_warm_fac)-num_unary
    binary_signals = add_depth_binary(base_pop,num_binary)
    unary_signals = add_depth_unary(base_pop,num_unary)

    return binary_signals + unary_signals

def warmstart_begin(trees):
    '''
    Performs warmstart and returns the first generation of alphas
    by generating a larger population size than required
    and randomly selects n individuals out of it
    '''
    warm_trees = tree_creation_begin(trees)
    random_population = global_random_state.sample(range(len(warm_trees)),num_individuals)
    fin_strategy_tree = [warm_trees[i] for i in random_population]
    return fin_strategy_tree


## Version 2

In [ ]:
def tree_creation_advanced(multiplicative_factor,base_pop):
    ''''
    Generates the unary and binary rooted trees for the warmstart function.
    Here we deliberately keep the number of unary population very less.
    '''
    count = int(num_individuals*multiplicative_factor)
    num_unary = global_np_random_state.randint(0,max(1,count//5))
    num_binary = count-num_unary
    num_unary=max(num_unary,0)
    num_binary=max(num_binary,0)
    if num_binary==0:
      return base_pop
    binary_signals = add_depth_binary(base_pop,num_binary)
    unary_signals = add_depth_unary(base_pop,num_unary)

    return binary_signals + unary_signals

def warmstart_advanced(prev_trees,trees,multiplicative_factor):
    '''
    Performs warmstart and returns the generation of alphas for datasets other than the 1st one by generating a larger population size
    than required and randomly selecting n individuals out of it
    '''
    check_count=int(num_individuals*multiplicative_factor)
    if(multiplicative_factor<=0.01):
      return prev_trees
    warm_trees= tree_creation_advanced(multiplicative_factor,trees)
    warm_trees=warm_trees+prev_trees
    if(num_individuals>len(warm_trees)):
        population_size=len(warm_trees)
    else:
      population_size=num_individuals
    random_population = global_random_state.sample(range(len(warm_trees)), population_size)
    fin_strategy_tree=[warm_trees[i] for i in random_population]
    return fin_strategy_tree

# Integration

## Version 1

In [ ]:
def best_strategy(dataset,base_signals,prev_trees,depth,ishigh,**kwargs):

  """
  Finds the best strategy at a specific depth by initializing with a warm start and
  building higher generations iteratively.
  Args:
      dataset:the array of datasets
      base_signals: array of corresponding base signals
      base_trees: array of corresponding trees
      depth: the max depth
  Returns:
        best trees and their fitness for the given depth
  """
  warm_strategy_pop= warmstart_begin(prev_trees)
  strategy_optimal=warm_strategy_pop.copy()
  fitness_prev_arr, fitness_prev_pnl_arr = [], []
  best_fit, prev_fit,count= 0,0,1

  #Initial fitness evaluation
  tree_pop_new_fit_arr=[]
  for index,tree in enumerate(warm_strategy_pop):
    final_signal=np.array([tree_signal(base_signals[0],tree)][0])
    final_signal = np.where(final_signal >signal_threshold, 1, np.where(final_signal < -signal_threshold, -1, 0))
    tree_pop_new_fit_arr.append(final_signal)
  obj = VectorBacktest(dataset[0],tree_pop_new_fit_arr)
  metrics=obj.port_ret()
  fitness_prev_pnl_arr=[]
  for i,col in enumerate((metrics.value().columns)):
    val=(metrics.value()[col].values)
    fitness_prev_pnl_arr.append(val)
  fitness_prev_arr=[(0 if x > 200 else max(x,-200.0), ind) for ind,x in enumerate(obj.fitness())]

  fitness_prev_arr_copy = fitness_prev_arr.copy()
  fitness_prev_pnl_arr=np.array(fitness_prev_pnl_arr)
  initial_similarity_matrix = calculate_similarity_matrix_np(fitness_prev_pnl_arr)
  cleaned_similarity_matrix, counts = analyze_similarity(initial_similarity_matrix)
  for fit_val, index in fitness_prev_arr:
    count = counts[index]
    fitness_prev_arr_copy[index] = (fit_val / (1.0 + count), index)
  fitness_prev_arr = fitness_prev_arr_copy

  #Average fitness of top 10% strategies
  prev_fit = np.mean([t[0] for t in sorted(fitness_prev_arr, key=lambda x: x[0], reverse = True)[:num_elite]])

  #Distributed Evolution loop
  tot_dataset_len=sum([len(d) for d in dataset])
  generation_arr=[(num_generations*len(d))//(tot_dataset_len)+1 for d in dataset]

  if not kwargs.get("prev_cross_rate"):
    kwargs["prev_cross_rate"]=ini_prev_cross
    kwargs["curr_cross_rate"]=ini_curr_cross
    kwargs["prev_cross_mom"]=ini_prev_cross_mom
    kwargs["prev_cross_vel"]=ini_prev_cross_vel
    kwargs["prev_mut_rate"]=ini_prev_mut
    kwargs["curr_mut_rate"]=ini_curr_mut
    kwargs["prev_mut_mom"]=ini_prev_mut_mom
    kwargs["prev_mut_vel"]=ini_prev_mut_vel
    kwargs["beta1"]=beta1
    kwargs["beta2"]=beta2
    kwargs["eta"]=eta
    if not kwargs.get("dataset_iteration"):
      kwargs["dataset_iteration"]=0
  tot_generations=0
  display_gen=0
  for j,dist_generation in enumerate(generation_arr):
    for gen in range(1,max(1,dist_generation)+1):
      print(f"########generation {display_gen+gen} started################")
      warm_strategy_pop,next_gen_fit, fitness_prev_arr, fitness_prev_pnl_arr,kwargs = simulated_next_generation(base_signals[j],
                                                                                                                warm_strategy_pop,
                                                                                                                dataset[j],
                                                                                                                gen,
                                                                                                                dist_generation,
                                                                                                                fitness_prev_arr,
                                                                                                                depth,
                                                                                                                ishigh,
                                                                                                                **kwargs)
      if next_gen_fit > best_fit:
        best_fit, strategy_optimal= next_gen_fit,warm_strategy_pop.copy()

      #Early stopping if fitness improvement falls below threshold
      if abs(next_gen_fit - prev_fit) <=stop_threshold:
        count += 1
        if count>stopping_generation:
          break
      else:
        count = 1
      prev_fit = next_gen_fit
      tot_generations+=1
    display_gen=tot_generations
  return strategy_optimal,best_fit,kwargs



def integrator(dataset, base_signals,base_trees,ishigh,**kwargs):
  '''
  Integrates the optimization algorithm across multiple depths, building and optimizing
  strategies at each depth level.For each volatility regime we evolve the population through
  G generations dividing the number of generations into each dataset proportionate to their size.
  Args:
      dataset:the array of datasets
      base_signals: array of corresponding base signals
      base_trees: array of corresponding trees
      depth: the max depth
  Returns:
      The dictionary of depthwise strategy trees and their fitnesses
  '''
  if not base_trees or len(base_trees) == 0:
        raise ValueError("base_trees must contain at least one tree")
  strategy_optimal= base_trees[0]
  depth_dict= {}  #Stores results by addition of depth to the existing trees
  for d in range(2, num_depth + 1):
    strategy_optimal,best_fit,kwargs= best_strategy(dataset,base_signals,strategy_optimal,d,ishigh,**kwargs)
    depth_dict[d]={
        'best_fit': best_fit,
        'tree_opt':strategy_optimal,
        'prev_cross':kwargs["prev_cross_rate"],
        'prev_cross_mom':kwargs["prev_cross_mom"],
        'prev_cross_vel':kwargs["prev_cross_vel"],
        'curr_cross':kwargs["curr_cross_rate"],
        'prev_mut':kwargs["prev_mut_rate"],
        'prev_mut_mom':kwargs["prev_mut_mom"],
        'prev_mut_vel':kwargs["prev_mut_vel"],
        'curr_mut':kwargs["curr_mut_rate"],
        'dataset_iteration':kwargs["dataset_iteration"]
    }
    print(f"##*****************DEPTH {d} has BEST_FITNESS of {best_fit}***********************##")

  return depth_dict


## Version 2

In [ ]:
def best_strategy_advanced(dataset,base_signals,prev_trees,
                           warmstart_percent,warmstart_tree,
                           best_fit,
                           depth,
                           ishigh,
                           **kwargs):
  """
  Finds the best strategy at a specific depth by initializing without
  a warmstart and building higher generations iteratively.
  Args:
      dataset:the array of datasets
      base_signals: array of corresponding base signals
      base_trees: array of corresponding trees
      depth: the max depth
  Returns:
      Best trees and their fitness for the given depth
  """
  warm_strategy_pop= warmstart_advanced(prev_trees.copy(),warmstart_tree.copy() ,warmstart_percent)
  dataset_iter=kwargs["dataset_iteration"]
  strategy_optimal=warm_strategy_pop.copy()
  fitness_prev_arr, fitness_prev_pnl_arr = [], []
  prev_fit, count= 0,1
  strategy_optimal= warm_strategy_pop.copy()
  #Initial fitness evaluation
  tree_pop_new_fit_arr=[]
  for index, tree in enumerate(warm_strategy_pop):
    final_signal=np.array([tree_signal(base_signals[0],tree)][0])
    final_signal = np.where(final_signal >signal_threshold, 1, np.where(final_signal < -signal_threshold, -1, 0))
    tree_pop_new_fit_arr.append(final_signal)
  obj = VectorBacktest(dataset[0],tree_pop_new_fit_arr)
  metrics=obj.port_ret()
  fitness_prev_pnl_arr=[]
  for i,col in enumerate((metrics.value().columns)):
    val=(metrics.value()[col].values)
    fitness_prev_pnl_arr.append(val)
  fitness_prev_arr=[(0 if x > 200 else max(x,-200.0), ind) for ind,x in enumerate(obj.fitness())]

  fitness_prev_pnl_arr=np.array(fitness_prev_pnl_arr)
  initial_similarity_matrix = calculate_similarity_matrix_np(fitness_prev_pnl_arr)
  cleaned_similarity_matrix, counts = analyze_similarity(initial_similarity_matrix)
  #Penalize similar chromosomes for diversity
  for fit_val, index in fitness_prev_arr:
    cnt = counts[index]
    fitness_prev_arr[index] = (fit_val/(1.0+cnt), index)

  #Sort chromosomes by fitness(taking top 10)
  prev_fit = np.mean([t[0] for t in sorted(fitness_prev_arr, key=lambda x: x[0], reverse = True)[:num_elite]])

  #Distributed Evolution loop
  tot_dataset_len=sum([len(d) for d in dataset])
  iter_arr=[(num_generations*len(d))//(tot_dataset_len)+1 for d in dataset]
  tot_it=0
  display_gen=0
  for j,dist_it in enumerate(iter_arr):
    dist_it=max(dist_it,1)
    for it in range(1,max(dist_it,1)+1):
      print(f"generation {display_gen+it} started!!")
      warm_strategy_pop,next_gen_fit, fitness_prev_arr, fitness_prev_pnl_arr,kwargs= simulated_next_generation(base_signals[j],
                                                                                                      warm_strategy_pop,dataset[j],
                                                                                                      it, dist_it, fitness_prev_arr,
                                                                                                      depth,ishigh,
                                                                                                      **kwargs)
      if next_gen_fit > best_fit:
        best_fit, strategy_optimal= next_gen_fit,warm_strategy_pop

      #Early stopping if fitness improvement falls below threshold
      if abs(next_gen_fit - prev_fit) <= stop_threshold:
        count += 1
        if count>stopping_generation:
          break
      else:
        count = 1
      prev_fit = next_gen_fit
      tot_it+=1
    display_gen=tot_it
  return strategy_optimal,best_fit,kwargs


def integrator_advanced(dataset, base_signals,new_trees,depth,
                        best_fit,warmstart_percent,warmstart_tree,ishigh,
                        **kwargs):
  """
  Integrates the optimization algorithm across multiple depths, building and optimizing
  strategies at each depth level.For each volatility regime we evolve the population through
  G generations dividing the number of generations into each dataset proportionate to their size.
  Args:
      dataset:the array of datasets
      base_signals: array of corresponding base signals
      base_trees: array of corresponding trees
      depth: the max depth
  Returns:
      The dictionary of depthwise strategy trees and their fitnesses
  """
  strategy_optimal= new_trees[0] #Stores results by depth
  strategy_optimal,best_fit,kwargs=best_strategy_advanced(dataset,base_signals,strategy_optimal,
                                                           warmstart_percent,warmstart_tree,best_fit,
                                                           depth,ishigh,**kwargs)
  dicti={
        'best_fit': best_fit,
        'tree_opt': strategy_optimal,
        'prev_cross':kwargs["prev_cross_rate"],
        'prev_cross_mom':kwargs["prev_cross_mom"],
        'prev_cross_vel':kwargs["prev_cross_vel"],
        'curr_cross':kwargs["curr_cross_rate"],
        'prev_mut':kwargs["prev_mut_rate"],
        'prev_mut_mom':kwargs["prev_mut_mom"],
        'prev_mut_vel':kwargs["prev_mut_vel"],
        'curr_mut':kwargs["curr_mut_rate"],
        'dataset_iteration':kwargs["dataset_iteration"]
    }
  print(f"################### DEPTH {depth} has BEST_FITNESS of {best_fit} #######################")
  return dicti


# Final Pipeline Execution

## Train-Validation Execution

In [ ]:
pred_vol_arr,skip_dataset,timeperiod_based_top=[],{2:0,3:0,4:0},{}

#Shift training-test window for walk-forward analysis
def shifter(train_start,train_end,test_start,test_end):
  train_start+=sliding_window
  train_end+=sliding_window
  test_start=train_end
  test_end+=sliding_window
  return train_start,train_end,test_start,test_end

#Main Function
def run_training_loop(df):
    global train_start,train_end,test_start,test_end,dataset_iteration
    global dict_high, dict_low,curr_warmstart_percent
    while test_end< len(df):
        #Build training dataset
        (train_df,train_dataset,predict_vol,garch_result,
         high_vol_dataset,low_vol_dataset,start_high_ind,
         end_high_ind,start_low_ind,end_low_ind) = prepare_training_data(df)

        #Evolutionary training
        if (dataset_iteration==0):
            dict_high, dict_low = initial_evolution_training(train_df,start_high_ind,end_high_ind,
                                                             start_low_ind,end_low_ind,
                                                             high_vol_dataset, low_vol_dataset)
        else:
            dict_high, dict_low = continued_evolution_training(train_df,start_high_ind,end_high_ind,
                                                               start_low_ind,end_low_ind,dict_high,dict_low,
                                                               high_vol_dataset,low_vol_dataset)
            curr_warmstart_percent *= warmstart_percent

        #Rolling GARCH forecasting
        pred_volatility = perform_rolling_garch_forecast(train_df, garch_result, df[test_start:test_end])

        #Volatility classification for test dataset
        final_classified_vol = classify_volatility(predict_vol, pred_volatility)
        pred_vol_arr.append(final_classified_vol)

        #Generate signals from evolved trees
        base_signals = dataset_preprocess(df[test_start:test_end],list(df.columns[start_col:(start_col+num_indicators)]),
                                          0, 0, istest=True)

        #Evaluate signals and store metrics
        evaluate_signals(df[test_start:test_end],final_classified_vol,
                         base_signals,dict_high,dict_low)

        #Shift the window
        shift_windows()
        if dataset_iteration> 20: #num of walkforward window
            break

#Prepare Training Data
def prepare_training_data(df):
    train_df= df[train_start:train_end]
    train_dataset= pd.DataFrame(train_df[['Close']])

    (garch_result,predict_vol,high_vol_dataset,low_vol_dataset,
     start_high_ind,end_high_ind,start_low_ind,end_low_ind)=volatility_classifier(train_dataset)

    return (train_df,train_dataset,predict_vol,garch_result,
            high_vol_dataset,low_vol_dataset,start_high_ind,
            end_high_ind,start_low_ind,end_low_ind)

#Initital Evolution
def initial_evolution_training(train_df,start_high_ind,end_high_ind,start_low_ind,
                               end_low_ind,high_vol_dataset,low_vol_dataset):
    high_base_trees, high_base_signals = [], []
    low_base_trees, low_base_signals = [], []

    #High vol base strategies
    for i in range(len(start_high_ind)):
        t,s = dataset_preprocess(train_df,list(train_df.columns[start_col:(start_col+num_indicators)]),
                                 start_high_ind[i],end_high_ind[i],isfirst=True)
        high_base_trees.append(t)
        high_base_signals.append(s)

    #Low vol base strategies
    for i in range(len(start_low_ind)):
        t,s = dataset_preprocess(train_df,list(train_df.columns[start_col:(start_col+num_indicators)]),
                                 start_low_ind[i],end_low_ind[i],isfirst=True)
        low_base_trees.append(t)
        low_base_signals.append(s)

    #Integrate
    dict_low = integrator(low_vol_dataset, low_base_signals, low_base_trees, ishigh=False)
    dict_high = integrator(high_vol_dataset, high_base_signals, high_base_trees, ishigh=True)
    return dict_high, dict_low



#Continued Evolution On New Walkforward Windows
def continued_evolution_training(train_df,start_high_ind,end_high_ind,start_low_ind,
                                 end_low_ind,dict_high,dict_low,
                                 high_vol_dataset,low_vol_dataset):

    for index in range(2, num_depth + 1):
        high_base_trees,high_base_signals= [], []
        low_base_trees,low_base_signals= [], []

        #Build signals for continued training
        for i in range(len(start_high_ind)):
            signals= dataset_preprocess(train_df,list(train_df.columns[start_col:(start_col+num_indicators)]),
                                         start_high_ind[i], end_high_ind[i])
            high_base_trees.append(dict_high[index]["tree_opt"])
            high_base_signals.append(signals)

        for i in range(len(start_low_ind)):
            signals= dataset_preprocess(train_df,list(train_df.columns[start_col:(start_col+num_indicators)]),
                                        start_low_ind[i],end_low_ind[i])
            low_base_trees.append(dict_low[index]["tree_opt"])
            low_base_signals.append(signals)

        kwargs_low = {"prev_mut_vel": ini_prev_mut_vel,
                      "prev_cross_rate": dict_low[index]["prev_cross"],
                      "curr_cross_rate": dict_low[index]["curr_cross"],
                      "prev_cross_mom": ini_prev_cross_mom,
                      "prev_mut_rate": dict_low[index]["prev_mut"],
                      "curr_mut_rate": dict_low[index]["curr_mut"],
                      "prev_mut_mom": ini_prev_mut_mom,
                      "prev_cross_vel": ini_prev_cross_vel,
                      "dataset_iteration": dict_low[index]["dataset_iteration"] + 1}

        kwargs_high = {"beta1": beta1,
                        "beta2": beta2,
                        "eta": eta,
                        "prev_mut_vel": ini_prev_mut_vel,
                        "prev_cross_rate": dict_high[index]["prev_cross"],
                        "curr_cross_rate": dict_high[index]["curr_cross"],
                        "prev_cross_mom": ini_prev_cross_mom,
                        "prev_mut_rate": dict_high[index]["prev_mut"],
                        "curr_mut_rate": dict_high[index]["curr_mut"],
                        "prev_mut_mom": ini_prev_mut_mom,
                        "prev_cross_vel": ini_prev_cross_vel,
                        "dataset_iteration": dict_high[index]["dataset_iteration"]+1}

        #Warmstart
        if index> 2:
            warmstart_high= dict_high[index- 1]["tree_opt"]
            warmstart_low= dict_low[index- 1]["tree_opt"]
        else:
            warmstart_high= [TreeNode(i) for i in range(num_indicators)]
            warmstart_low= [TreeNode(i) for i in range(num_indicators)]

        #High Vol update
        print(f"High vol dataset training started for next {dataset_iteration}")
        if high_base_trees:
            dict_high[index]= integrator_advanced(high_vol_dataset,high_base_signals,high_base_trees,
                                                   index,dict_high[index]["best_fit"],curr_warmstart_percent,
                                                  warmstart_high,ishigh=True,**kwargs_high)
        else:
            dict_high[index]= {
                'best_fit': dict_high[index]["best_fit"],'tree_opt': dict_high[index]["tree_opt"],
                'prev_cross':kwargs_high["prev_cross_rate"],'prev_cross_mom':kwargs_high["prev_cross_mom"],
                'prev_cross_vel':kwargs_high["prev_cross_vel"],'curr_cross':kwargs_high["curr_cross_rate"],
                'prev_mut':kwargs_high["prev_mut_rate"],'prev_mut_mom':kwargs_high["prev_mut_mom"],
                'prev_mut_vel':kwargs_high["prev_mut_vel"],'curr_mut':kwargs_high["curr_mut_rate"],
                'dataset_iteration':kwargs_high["dataset_iteration"]}

        #Low Vol update
        print(f"Low vol dataset training started for next {dataset_iteration}")
        if low_base_trees:
            dict_low[index] = integrator_advanced(low_vol_dataset,low_base_signals,low_base_trees,
                                                  index,dict_low[index]["best_fit"],curr_warmstart_percent,
                                                  warmstart_low,ishigh=False,**kwargs_low)
        else:
            dict_low[index] = {'best_fit': dict_low[index]["best_fit"],'tree_opt': dict_low[index]["tree_opt"],
                                'prev_cross':kwargs_low["prev_cross_rate"],'prev_cross_mom':kwargs_low["prev_cross_mom"],
                                'prev_cross_vel':kwargs_low["prev_cross_vel"],'curr_cross':kwargs_low["curr_cross_rate"],
                                'prev_mut':kwargs_low["prev_mut_rate"],'prev_mut_mom':kwargs_low["prev_mut_mom"],
                                'prev_mut_vel':kwargs_low["prev_mut_vel"],'curr_mut':kwargs_low["curr_mut_rate"],
                                'dataset_iteration':kwargs_low["dataset_iteration"]}
    return dict_high, dict_low


#Rolling GARCH forecasting
def perform_rolling_garch_forecast(train_df, garch_result, test_df):

    train_returns= train_df['Close'].pct_change().dropna()*100
    test_returns= test_df['Close'].pct_change().dropna()*100
    garch_params= {'p': garch_result.model.volatility.p,
                    'q': garch_result.model.volatility.q,
                    'o': garch_result.model.volatility.o,
                    'power': garch_result.model.volatility.power,
                    'dist': "StudentsT"}
    all_forecasts= []
    step_size= 10 #future forecasting days
    for i in range(0,len(test_returns),step_size):
        current_train = pd.concat([train_returns, test_returns.iloc[:i]])
        model = arch_model(current_train, **garch_params)
        res = model.fit(disp='off', show_warning=False)
        horizon = min(step_size, len(test_returns) - i)
        if horizon <= 0:
            break

        forecast = res.forecast(horizon=horizon, reindex=False)
        variance_chunk = forecast.variance.values[-1, :]
        all_forecasts.append(np.sqrt(variance_chunk))

    pred_volatility = np.concatenate(all_forecasts)
    return pred_volatility[:len(test_df)]



#Volatility Classification for test dataset
def classify_volatility(predict_vol, pred_volatility):
    combined= np.concatenate([predict_vol, pred_volatility])
    fixed_len= len(predict_vol)

    classified= np.zeros_like(pred_volatility, dtype=int)
    window= 30
    for i in range(fixed_len, len(combined)):
        m = np.mean(combined[i - window : i])
        idx = i - fixed_len
        classified[idx] = int(pred_volatility[idx] > m)

    #First day classification
    train_mean = np.mean(predict_vol[-window:])
    first_day = int(pred_volatility[0] > train_mean)
    return np.insert(classified, 0, first_day)



#Signal Generation
def generate_test_signals(base_signals, final_vol_class, high_trees, low_trees):
    signals = []
    high_signals = [test_signal_generator(t, base_signals) for t in high_trees]
    low_signals = [test_signal_generator(t, base_signals) for t in low_trees]
    for i in range(max(len(high_signals),len(low_signals))):
        for j in range(max(len(low_signals),len(high_signals))):
            if high_signals and low_signals:
                s= np.where(final_vol_class, high_signals[i], low_signals[j])
            elif high_signals:
                s= high_signals[i]
            else:
                s= low_signals[j]
            signals.append(s)
    return signals



#Evaluate Signals
def evaluate_signals(test_dataset, final_vol_class, base_signals, dict_high, dict_low):
    global timeperiod_based_top, avg_test_res, avg_sharpe_dict, dataset_iteration
    timeperiod_based_top[dataset_iteration] = []

    for index in range(2, num_depth + 1):
        high_trees = dict_high[index]["tree_opt"]
        low_trees = dict_low[index]["tree_opt"]

        test_signal_arr = generate_test_signals(base_signals, final_vol_class, high_trees, low_trees)
        bt = VectorBacktest(test_dataset, test_signal_arr)
        metrics = bt.port_ret()

        #calculating the metrics of sharpe,profit over the duration and maximum drawdown
        sharpe_arr = metrics.sharpe_ratio()
        ret_arr = metrics.total_profit()
        mdd_arr = metrics.max_drawdown()

        sorted_sharpe,detail= [],[]

        max_ret= -math.inf
        best_sharpe= 0
        for i,sp in enumerate(sharpe_arr):
            if sp<-200 or sp>200:#illegitimate signals
                continue
            sorted_sharpe.append(sp)
            detail.append((sp, ret_arr[i], mdd_arr[i]))
            if ret_arr[i] > max_ret:
                max_ret = ret_arr[i]
                best_sharpe = sp
        sorted_sharpe.sort(reverse=True)
        detail.sort(reverse=True)
        avg_sharpe = np.mean(sorted_sharpe[:10])

        timeperiod_based_top[dataset_iteration].extend(detail[:10])
        avg_test_res[index] += sorted_sharpe[0]
        avg_sharpe_dict[index].append(avg_sharpe)
        print(f"Depth {index}: Best Sharpe={best_sharpe}, Return={max_ret}, Avg Sharpe={avg_sharpe}")
    timeperiod_based_top[dataset_iteration].sort(reverse=True)
    timeperiod_based_top[dataset_iteration] = timeperiod_based_top[dataset_iteration][:10]



#Shift train-test Window
def shift_windows():
    global train_start, train_end, test_start, test_end, dataset_iteration
    train_start, train_end, test_start, test_end = shifter(
        train_start, train_end, test_start, test_end
    )
    dataset_iteration += 1
if __name__ == "__main__":
    run_training_loop(df)

## Out-Of-Sample Test Execution Block

In [ ]:
def oos_tester(train_start, train_end, oos_shift, oos_depth):

    #Prepare training and GARCH model
    (train_df,garch_result,predict_vol,test_start,
     test_end)= prepare_oos_training_data(train_start, train_end, oos_shift)

    #Run rolling GARCH forecast
    (test_df, test_dataset,pred_volatility,train_returns,
     test_returns)= perform_oos_garch_forecast(train_df, garch_result, test_start, test_end)

    #Do regime classification
    final_classified_vol = classify_oos_volatility(predict_vol,pred_volatility)

    #Base indicators for signal generation
    base_signals = dataset_preprocess(test_df,list(df.columns[start_col:(start_col + num_indicators)]),
                                      0,0,istest=True)

    #Evaluate all evolved strategies out-of-sample
    metrics,fit_metric = evaluate_oos_metrics(base_signals,final_classified_vol,
                                              test_dataset,oos_depth)

    #Ranking strategies by Sharpe
    av_sp = []
    max_profit = -200
    best_ind = -1

    for i,prof in enumerate(fit_metric):
        if fit_metric[i]<-200 or fit_metric[i]> 200:
            continue
        av_sp.append((
            float(fit_metric[i]),
            float(metrics.total_profit()[i]),
            float(metrics.max_drawdown()[i])
        ))
        if prof> max_profit:
            max_profit= prof
            best_ind= i
    av_sp.sort(reverse=True)
    count_fin_strat = 10
    return av_sp[:count_fin_strat]

def prepare_oos_training_data(train_start, train_end, oos_shift):

    train_df = df[train_start:train_end]
    train_dataset = pd.DataFrame(train_df[['Close']])
    garch_result, predict_vol, _, _, _, _, _, _ = volatility_classifier(train_dataset)
    test_start = train_end
    test_end = len(df) - oos_shift

    #print(test_start, test_end)
    return train_df, garch_result, predict_vol, test_start, test_end

def perform_oos_garch_forecast(train_df, garch_result, test_start, test_end):

    test_df= df[test_start:test_end]
    test_dataset= test_df[['Close']].copy()
    train_returns= train_df['Close'].pct_change().dropna() * 100
    test_returns= test_df['Close'].pct_change().dropna() * 100

    garch_params= {'p': garch_result.model.volatility.p,'q': garch_result.model.volatility.q,
                    'o': garch_result.model.volatility.o,'power': garch_result.model.volatility.power,'dist': "StudentsT"}

    all_forecasts= []
    step_size= 10

    for i in range(0,len(test_returns),step_size):
        current_train_returns= pd.concat([train_returns, test_returns.iloc[:i]])
        model = arch_model(current_train_returns, **garch_params)
        res = model.fit(disp='off', show_warning=False)

        horizon = min(step_size, len(test_returns) - i)
        if horizon <= 0:
            break

        forecast = res.forecast(horizon=horizon, reindex=False)
        variance_chunk = forecast.variance.values[-1, :]
        all_forecasts.append(np.sqrt(variance_chunk))

    pred_volatility = np.concatenate(all_forecasts)
    pred_volatility = pred_volatility[:len(test_df)]

    return test_df, test_dataset, pred_volatility, train_returns, test_returns


def classify_oos_volatility(predict_vol, pred_volatility):

    combined_vol = np.concatenate([predict_vol, pred_volatility])
    fixed_train_len = len(predict_vol)
    test_vanilla_window = 30

    classified_vol = np.zeros_like(pred_volatility, dtype=int)

    for i in range(fixed_train_len, len(combined_vol)):
        roll_mean = np.mean(combined_vol[i - test_vanilla_window : i])
        idx = i - fixed_train_len
        if pred_volatility[idx] > roll_mean:
            classified_vol[idx] = 1

    final_train_window_mean = np.mean(predict_vol[-test_vanilla_window:])
    first_day = int(pred_volatility[0] > final_train_window_mean)

    return np.insert(classified_vol, 0, first_day)


def generate_oos_signals(high_trees, low_trees, base_signals, final_classified_vol):

    high_final = [test_signal_generator(t, base_signals) for t in high_trees]
    low_final  = [test_signal_generator(t, base_signals) for t in low_trees]

    test_signal_arr = []

    for i in range(max(len(high_final), len(low_final))):
        for j in range(max(len(low_final), len(high_final))):

            if high_final and low_final:
                sig = np.where(final_classified_vol, high_final[i], low_final[j])
            elif high_final:
                sig = high_final[i]
            elif low_final:
                sig = low_final[j]
            else:
                sig = np.zeros_like(final_classified_vol)

            test_signal_arr.append(sig)

    return test_signal_arr


def evaluate_oos_metrics(base_signals, final_classified_vol, test_dataset, oos_depth):

    def out_of_sample_metrics(high_ind, low_ind):

        m1 = time.time()
        high_trees = dict_high[high_ind]["tree_opt"]
        low_trees = dict_low[low_ind]["tree_opt"]

        signals = generate_oos_signals(
            high_trees, low_trees,
            base_signals, final_classified_vol
        )

        backtest = VectorBacktest(test_dataset, signals)
        elapsed = time.time() - m1
        print(f"Backtest complete in {elapsed:.3f} seconds\n")

        return backtest.port_ret(), backtest.fitness()

    return out_of_sample_metrics(oos_depth, oos_depth)

if __name__ == "__main__":
  train_start-=150
  train_end-=150
  best_arr,best_arr_1,best_arr_2={},{},{}
  for i in range(0,200,50):
    best_arr[i]=oos_tester(train_start,train_end,i,2)
    best_arr_1[i]=oos_tester(train_start,train_end,i,3)
    best_arr_2[i]=oos_tester(train_start,train_end,i,4)


# Print Tree

In [ ]:
def print_tree_compact(root):
    if not root:
        print("Empty tree")
        return
    def build_tree_string(node, prefix="", is_left=True):
        if node is None:
            return []
        result = []

        #Process right child first
        if node.right:
            extension = "│   " if is_left else "    "
            result.extend(build_tree_string(node.right, prefix + extension, False))
        #Add current node
        line = prefix + ("└── " if is_left else "┌── ") + str(node.val)
        result.append(line)
        #Process left child
        if node.left:
            extension = "    " if is_left else "│   "
            result.extend(build_tree_string(node.left, prefix + extension, True))
        return result

    tree_lines = build_tree_string(root, "", True)
    for line in tree_lines:
        print(line)

# Comparative Plot

## XGBLightGBM Results

In [ ]:
import warnings
import os
import math
import json
import logging
import itertools
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Tuple, Any,cast
from joblib import Parallel, delayed
from tqdm import tqdm
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")


In [ ]:
class VectorBacktestXGBLGB:
    def __init__(self, dataset, final_signals):
        self.df = dataset.copy()
        self.signal = np.array(final_signals)
        unique_values, counts = np.unique(self.signal, return_counts=True)
        self.df['Signal'] = self.signal.astype(int)

        self.positions = self.df['Signal'].shift(1).fillna(0)
        if (self.positions == 1).any():
            self.positions.iat[-1] = -1
        self.portfolio = vbt.Portfolio.from_signals(
            close=self.df['Close'],
            entries=(self.positions == 1),
            exits=(self.positions == -1),
            init_cash=init_cash,
            sl_stop=sl_stop,
            tp_stop=tp_stop,
            slippage=slippage/10,
            fees=fees/10,
            freq=freq
        )
        self.stats = self.portfolio.stats()

    def sharpe_capital(self):
        sharpe = self.stats[24]
        return sharpe
    def return_mdd(self):
        return (self.stats[5],self.stats[9])

#Logging setup
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True)
logger = logging.getLogger(__name__)

#Configurable parameters
DATA_PATH = "zscored_indicators_spy.csv"
OUTPUT_DIR = Path("model_search_outputs_csi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Walk-forward parameters (as requested)
WF_TRAIN_DAYS = 1000
WF_TEST_DAYS = 500
WF_STEP = 150

#Randomized search sizes (you can increase if you want more thorough search)
RANDOM_SEARCH_TRIALS = 40  # number of random trials per model per window
FOCUSED_GRID_TOP_K = 5     # take top K from random search to build focused grids
FOCUSED_GRID_RADIUS = {}

# Feature pruning
PRUNE_PERCENTILE = 10  # drop features below this importance percentile after randomized step
POS_THRESH = 0.01
NEG_THRESH = -0.01


In [ ]:
def add_forward_return(df: pd.DataFrame, horizon: int = 100, price_col: str = "Close", out_col: str = None) -> pd.DataFrame:
    """Add forward percent return column for a given horizon (shifted -horizon)."""
    if out_col is None:
        out_col = f"{horizon}_d_return"
    df = df.copy()
    df[out_col] = df[price_col].pct_change(horizon).shift(horizon)
    return df

def pick_indicator_columns(df: pd.DataFrame, indicator_prefix: str = None, start_col: int = None, num_ind: int = 80) -> List[str]:
    if indicator_prefix:
        cols = [c for c in df.columns if c.startswith(indicator_prefix)]
        if len(cols) >= num_ind:
            return cols[:num_ind]
        elif len(cols) > 0:
            return cols
    if start_col is not None:
        return list(df.columns[start_col:(start_col + num_ind)])
    exclude = {'Date', 'Close','High','Low','Open','Volume'}
    cols = [c for c in df.columns if c not in exclude]
    return cols[:num_ind]

def dist_preprocess_v2(df: pd.DataFrame, start_idx: int, end_idx: int, x: int, start_col: int):
    """Create base_trees placeholder and base_signals (unchanged semantic)."""
    base_columns = list(df.columns[start_col:(start_col + x)])
    base_signals = (df[base_columns][start_idx:end_idx].values.T)
    return base_signals

def preds_to_signals(preds: np.ndarray) -> np.ndarray:
    """Convert regression preds into discrete signals using the SAME thresholds you had."""
    return np.where(preds > POS_THRESH, 1, np.where(preds < NEG_THRESH, -1, 0))

def evaluate_model_with_backtest_v2(model, X_train, y_train, X_test, test_dataset):
    """Train model and return Sharpe from VectorBacktest (keeps original threshold logic)."""
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    signals = preds_to_signals(preds)

    # Run backtest
    try:
        bt = VectorBacktestXGBLGB(test_dataset, signals)
        sharpe = bt.sharpe_capital()
        if np.isnan(sharpe) or np.isinf(sharpe):
            return 0.0
        return sharpe
    except Exception as e:
        logger.warning(f"Backtest failed: {e}")
        return -999.0
def evaluate_model_with_backtest_v3(model, X_train, y_train, X_test, test_dataset,plot_fin=False):
    """Train model and return Sharpe from VectorBacktest (keeps original threshold logic)."""
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    signals = preds_to_signals(preds)

    try:
        bt = VectorBacktestXGBLGB(test_dataset, signals)
        if(plot_fin):
            return bt.portfolio
        sharpe = bt.sharpe_capital()
        ret,mdd=bt.return_mdd()
        if np.isnan(sharpe) or np.isinf(sharpe):
            return 0.0,0.0,0.0
        return sharpe,ret,mdd
    except Exception as e:
        logger.warning(f"Backtest failed: {e}")
        return -999.0,0.0,0.0

def sample_random_param_combos(param_grid: Dict[str, List[Any]], n_samples: int) -> List[Dict[str, Any]]:
    """Uniformly sample n_samples combos from param_grid (without replacement when possible)."""
    all_combos = list(itertools.product(*param_grid.values()))
    total = len(all_combos)
    if total <= n_samples:
        return [dict(zip(param_grid.keys(), combo)) for combo in all_combos]
    sampled = random.sample(all_combos, n_samples)
    return [dict(zip(param_grid.keys(), combo)) for combo in sampled]

def build_focused_grid_around(params: Dict[str, Any], param_options: Dict[str, List[Any]], radius_map: Dict[str, int]) -> List[Dict[str, Any]]:
    """For a chosen param set (from randomized search), build a small grid by varying numeric params"""
    focused_sets = []
    keys = list(params.keys())
    choices_per_key = []
    for k in keys:
        options = param_options[k]
        if isinstance(params[k], (int, float)) and len(options) > 1:
            idx = options.index(params[k]) if params[k] in options else 0
            r = radius_map.get(k, 1)
            low = max(0, idx - r)
            high = min(len(options) - 1, idx + r)
            choices_per_key.append(options[low:high + 1])
        else:
            choices_per_key.append([params[k]])
    for combo in itertools.product(*choices_per_key):
        focused_sets.append(dict(zip(keys, combo)))
    return focused_sets
lgb_params_grid = {
    "n_estimators": [200, 400, 800],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 8],
    "num_leaves": [15, 31, 63],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

xgb_params_grid = {
    "n_estimators": [200, 400, 800],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 8],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

# radius map for focused grid (how many neighbor indices to include)
FOCUSED_RADIUS_MAP = {
    "n_estimators": 1,
    "learning_rate": 1,
    "max_depth": 1,
    "num_leaves": 1,
    "subsample": 0,
    "colsample_bytree": 0,
}

In [ ]:
# --- Main Pipeline: load, preprocess, run walk-forward ---

def run_pipeline(
    data_path=DATA_PATH,
    horizon=100,
    indicator_prefix="f",
    start_col=5,
    num_ind=50,
    output_dir=OUTPUT_DIR
):
    xdf = pd.read_csv(data_path)
    xdf = add_forward_return(xdf, horizon=horizon, price_col="Close")
    target_col = f"{horizon}_d_return"

    xdf = xdf.dropna().reset_index(drop=True)
    train_start,train_end=0,1000
    indicator_cols = pick_indicator_columns(xdf, indicator_prefix=indicator_prefix, start_col=start_col, num_ind=num_ind)
    if len(indicator_cols) < num_ind:
        logger.warning(f"Only found {len(indicator_cols)} indicator columns (requested {num_ind}). Using what exists.")

    #prepare results storage
    per_window_records = []

    total_len = len(xdf)
    #walk-forward windows
    wf_count = 0
    last_best_lgb = None
    last_best_xgb = None
    last_kept_cols = indicator_cols
    print(train_start,train_end)
    while True:
        test_start = train_end
        test_end = test_start + WF_TEST_DAYS

        if test_end > 4801:
            break

        logger.info(f"WalkForward Window #{wf_count}: train [{train_start}:{train_end}] test [{test_start}:{test_end}]")

        #slice data
        train_df = xdf.iloc[train_start:train_end].reset_index(drop=True)
        test_df = xdf.iloc[test_start:test_end].reset_index(drop=True)

        X_train = train_df[indicator_cols].values
        y_train = train_df[target_col].values
        X_test = test_df[indicator_cols].values

        test_dataset = test_df[['Close']].copy()
        #Sample param combos
        lgb_random_samples = sample_random_param_combos(lgb_params_grid, RANDOM_SEARCH_TRIALS)
        xgb_random_samples = sample_random_param_combos(xgb_params_grid, RANDOM_SEARCH_TRIALS)

        def eval_combo_lgb(params):
            model = LGBMRegressor(**params, verbosity=-1, random_state=42)
            score = evaluate_model_with_backtest_v2(model, X_train, y_train, X_test, test_dataset)
            return params, score

        def eval_combo_xgb(params):
            model = XGBRegressor(**params, verbosity=0, random_state=42)
            score = evaluate_model_with_backtest_v2(model, X_train, y_train, X_test, test_dataset)
            return params, score
        logger.info("Randomized search LGBM...")
        lgb_rand_results = Parallel(n_jobs=2)(
            delayed(eval_combo_lgb)(p) for p in tqdm(lgb_random_samples, desc="LGBM Random")
        )

        logger.info("Randomized search XGB...")
        xgb_rand_results = Parallel(n_jobs=2)(
            delayed(eval_combo_xgb)(p) for p in tqdm(xgb_random_samples, desc="XGB Random")
        )

        #sort by Sharpe desc
        lgb_rand_results_sorted = sorted(lgb_rand_results, key=lambda x: x[1], reverse=True)
        xgb_rand_results_sorted = sorted(xgb_rand_results, key=lambda x: x[1], reverse=True)

        #Save randomized search results for this window
        rnd_out = {
            "window": wf_count,
            "train_idx": (train_start, train_end),
            "test_idx": (test_start, test_end),
            "lgb_random": lgb_rand_results_sorted,
            "xgb_random": xgb_rand_results_sorted,
        }
        rnd_path = output_dir / f"window_{wf_count}_random_results.json"
        with open(rnd_path, "w") as f:
            json.dump({
                "lgb_random": [(r[0], float(r[1])) for r in lgb_rand_results_sorted],
                "xgb_random": [(r[0], float(r[1])) for r in xgb_rand_results_sorted]
            }, f, default=str)

        #Train best models to extract feature importances
        lgb_best_params = lgb_rand_results_sorted[0][0] if lgb_rand_results_sorted else None
        xgb_best_params = xgb_rand_results_sorted[0][0] if xgb_rand_results_sorted else None
        lgb_best_score = lgb_rand_results_sorted[0][1] if lgb_rand_results_sorted else -999
        xgb_best_score = xgb_rand_results_sorted[0][1] if xgb_rand_results_sorted else -999

        # Decide which model's importances to use: choose higher Sharpe from random stage
        chosen_model_type = None
        chosen_model = None
        if lgb_best_score >= xgb_best_score and lgb_best_params is not None:
            chosen_model_type = "lgb"
            chosen_model = LGBMRegressor(**lgb_best_params, verbosity=-1, random_state=42).fit(X_train, y_train)
            importances = np.array(chosen_model.feature_importances_)
        elif xgb_best_params is not None:
            chosen_model_type = "xgb"
            chosen_model = XGBRegressor(**xgb_best_params, verbosity=0, random_state=42).fit(X_train, y_train)
            importances = np.array(getattr(chosen_model, "feature_importances_", np.zeros(X_train.shape[1])))
        else:
            importances = np.ones(X_train.shape[1])

        #compute threshold and prune
        cutoff = np.percentile(importances, PRUNE_PERCENTILE)
        keep_mask = importances > cutoff
        kept_cols = [c for i, c in enumerate(indicator_cols) if keep_mask[i]]
        dropped_cols = [c for i, c in enumerate(indicator_cols) if not keep_mask[i]]
        logger.info(f"Pruned features: dropped {len(dropped_cols)} / {len(indicator_cols)} (cutoff={cutoff:.6f})")
        if len(kept_cols) == 0:
            logger.warning("All features pruned! Reverting to original set.")
            kept_cols = indicator_cols.copy()

        #Recompute X matrices with reduced features
        X_train_pruned = train_df[kept_cols].values
        X_test_pruned = test_df[kept_cols].values
        #Take top K randomized param sets and build small focused grids
        def focused_search_for_model(model_name, rand_sorted, param_grid, eval_func, radius_map):
            focused_candidates = []
            top_k = rand_sorted[:FOCUSED_GRID_TOP_K]
            seen = set()
            for params, _score in top_k:
                focused = build_focused_grid_around(params, param_grid, radius_map)
                for f in focused:
                    key = tuple(sorted(f.items()))
                    if key not in seen:
                        seen.add(key)
                        focused_candidates.append(f)
            out = Parallel()(
                delayed(lambda p: (p, eval_func(p, X_train_pruned, y_train, X_test_pruned, test_dataset)))(p)
                for p in tqdm(focused_candidates, desc=f"{model_name} Focused")
            )
            return sorted(out, key=lambda x: x[1], reverse=True)

        #wrapper evaluation functions that accept params first
        def eval_lgb_params(params, X_tr, y_tr, X_te, test_ds):
            model = LGBMRegressor(**params, verbosity=-1, random_state=42)
            return evaluate_model_with_backtest_v2(model, X_tr, y_tr, X_te, test_ds)

        def eval_xgb_params(params, X_tr, y_tr, X_te, test_ds):
            model = XGBRegressor(**params, verbosity=0, random_state=42)
            return evaluate_model_with_backtest_v2(model, X_tr, y_tr, X_te, test_ds)

        lgb_focused_results = focused_search_for_model("LGBM", lgb_rand_results_sorted, lgb_params_grid, eval_lgb_params, FOCUSED_RADIUS_MAP)
        xgb_focused_results = focused_search_for_model("XGB", xgb_rand_results_sorted, xgb_params_grid, eval_xgb_params, FOCUSED_RADIUS_MAP)

        #Collect best result among all
        best_lgb = lgb_focused_results[0] if lgb_focused_results else (None, -999.0)
        best_xgb = xgb_focused_results[0] if xgb_focused_results else (None, -999.0)
        #Track last trained
        last_best_lgb = best_lgb
        last_best_xgb = best_xgb
        last_kept_cols = kept_cols
        if best_lgb[1] >= best_xgb[1]:
            chosen_final_model = "LGBM"
            best_params = best_lgb[0]
            best_sharpe = best_lgb[1]
        else:
            chosen_final_model = "XGB"
            best_params = best_xgb[0]
            best_sharpe = best_xgb[1]
        logger.info(f"Window {wf_count} lgb best model: {best_lgb[0]} Sharpe={best_lgb[1]:.4f} and xgb best model: {best_xgb[0]} Sharpe={best_xgb[1]:.4f}")
        logger.info(f"Window {wf_count} best model: {chosen_final_model} Sharpe={best_sharpe:.4f}, Params={best_params}")

        #Save focused results for window
        focused_out_path = output_dir / f"window_{wf_count}_focused_results.json"
        with open(focused_out_path, "w") as f:
            json.dump({
                "best_lgb": (best_lgb[0], float(best_lgb[1]) if best_lgb[1] is not None else None),
                "best_xgb": (best_xgb[0], float(best_xgb[1]) if best_xgb[1] is not None else None),
                "chosen_final_model": chosen_final_model,
                "chosen_params": best_params,
                "chosen_sharpe": float(best_sharpe)
            }, f, default=str)

        #record per-window summary
        per_window_records.append({
            "window": wf_count,
            "train_idx": (train_start, train_end),
            "test_idx": (test_start, test_end),
            "chosen_model": chosen_final_model,
            "chosen_params": best_params,
            "chosen_sharpe": float(best_sharpe),
            "n_original_features": len(indicator_cols),
            "n_pruned_features": len(kept_cols),
            "dropped_features": dropped_cols,
        })

        #slide window
        train_start += WF_STEP
        train_end += WF_STEP
        wf_count += 1
    extra_cuts = [0]
    train_start-=WF_STEP
    train_end-=WF_STEP
    for cut in extra_cuts:
        test_start = train_end
        test_end = len(df) - cut
        if test_end <= test_start:
            continue
        test_df = xdf.iloc[test_start:test_end].reset_index(drop=True)
        X_test_pruned = test_df[last_kept_cols].values
        test_dataset = test_df[['Unnamed: 0', 'Close']].copy()

        #Evaluate frozen LGBM
        if last_best_lgb and isinstance(last_best_lgb[0],dict):
            lgb_model = LGBMRegressor(**last_best_lgb[0], verbosity=-1, random_state=42)
            lgb_port = evaluate_model_with_backtest_v3(lgb_model, xdf.iloc[train_end-1000:train_end][last_kept_cols].values,
                                                         xdf.iloc[train_end-1000:train_end][target_col].values,
                                                         X_test_pruned, test_dataset,plot_fin=True)
            per_window_records.append({
                "window": f"extra_LGBM_cut{cut}",
                "train_idx": (train_end-1000,train_end),
                "test_idx": (test_start, test_end),
                "chosen_model": "LGBM",
                "chosen_params": last_best_lgb[0],
                "n_original_features": len(indicator_cols),
                "n_pruned_features": len(last_kept_cols)
            })

        if last_best_xgb and isinstance(last_best_xgb[0],dict):
            xgb_model = XGBRegressor(**last_best_xgb[0], verbosity=0, random_state=42)
            xgb_port = evaluate_model_with_backtest_v3(xgb_model, xdf.iloc[train_end-1000:train_end][last_kept_cols].values,
                                                         xdf.iloc[train_end-1000:train_end][target_col].values,
                                                         X_test_pruned, test_dataset,plot_fin=True)
            per_window_records.append({
                "window": f"extra_XGB_cut{cut}",
                "train_idx": (train_end-1000,train_end),
                "test_idx": (test_start, test_end),
                "chosen_model": "XGB",
                "chosen_params": last_best_xgb[0],
                "n_original_features": len(indicator_cols),
                "n_pruned_features": len(last_kept_cols)
            })
    return xgb_port,lgb_port,per_window_records


In [ ]:
xgb_port,lgb_port,summary = run_pipeline(
    data_path=DATA_PATH,
    horizon=100,
    indicator_prefix="f",
    start_col=5,
    num_ind=53,
    output_dir=OUTPUT_DIR
)

## GPlearn Results

In [ ]:
!pip install gplearn
import gplearn as gpl
import warnings

warnings.filterwarnings("ignore")

import sklearn
print(sklearn.__version__)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from gplearn.genetic import SymbolicTransformer,SymbolicRegressor
from gplearn.functions import make_function
from gplearn.fitness import make_fitness, _Fitness

In [ ]:
class VectorBacktestGP:
    def __init__(self, dataset, final_signals):
        self.df = dataset.copy()
        self.signal = np.array(final_signals)
        self.signal = np.where(self.signal >=0.01, 1, np.where(self.signal <-0.01, -1, 0))
        unique_values, counts = np.unique(self.signal, return_counts=True)
        self.df['Signal'] = self.signal.astype(int)
        self.positions = self.df['Signal'].shift(1).fillna(0)

        if (self.positions == 1).any():
            self.positions.iat[-1] = -1

        self.portfolio = vbt.Portfolio.from_signals(
            close=self.df['Close'],
            entries=(self.positions == 1),
            exits=(self.positions == -1),
            slippage=0.0001,
            fees=0.0001,
            init_cash=10_000,
            freq='1D'
        )

        # Store portfolio stats
        self.stats = self.portfolio.stats()
    def sharpe_capital(self):
        sharpe = self.stats[24]
        return sharpe
    def return_mdd(self):
        return (self.stats[5],self.stats[9])

In [ ]:
class CompGPlearn:
  def __init__(self,df:pd.DataFrame,st_day:int,train_len:int,test_len:int,sig_th:float,is_train:bool=True):
    self.df=df.iloc[st_day:st_day+train_len+test_len].copy().reset_index(drop=True)
    self.train_df=self.df.iloc[:train_len].copy().reset_index(drop=True)
    self.test_df=self.df.iloc[train_len:].copy().reset_index(drop=True)
    self.train_dataset=self.train_df[['Close']].reset_index(drop=True)
    self.test_dataset=self.test_df[['Close']].reset_index(drop=True)
    self.alpha_signals=self.dist_preprocess(self.df,0,train_len+test_len,num_ind) # Adjusted length

    self.X_train=np.nan_to_num(self.alpha_signals.T)[:train_len]
    self.X_test=np.nan_to_num(self.alpha_signals.T)[train_len:]
    self.y_train=np.nan_to_num(pd.DataFrame(df['fwd_return']).iloc[st_day:st_day+train_len].values)
    self.signal_threshold=sig_th
    self.custom_fitness = _Fitness(function=self.custom_fitness_,greater_is_better=True)
    self.function_set = ['add', 'sub', 'mul','abs', 'max', 'min']
    self.population_size = 500
    self.generations = 30
    self.random_state = 25
    self.est_gp=SymbolicRegressor(
        population_size=1000,
        generations=5,
        init_depth=(2, 6),
        tournament_size=600,
        stopping_criteria=1.,
        p_crossover=0.3,
        p_subtree_mutation=0.1,
        p_hoist_mutation=0.01,
        p_point_mutation=0.1,
        p_point_replace=0.6,
        max_samples=0.9,
        verbose=1,
        parsimony_coefficient=0.,
        random_state=self.random_state,
        function_set=self.function_set,
        metric=self.custom_fitness,
        const_range=None,
        n_jobs=1)
    if is_train:
      self.train_gplearn()
      self.final_generation,self.test_scores=self.test_gplearn()
      self.test_scores.sort(key=lambda x: x[0], reverse=True)
      self.test_sharpe=self.finalised_result()
  @staticmethod
  def dist_preprocess(df,start_idx,end_idx,x = 82,start_col=5):
    base_columns = list(df.columns[start_col:(start_col + x)])
    base_signals=(df[base_columns][start_idx:end_idx].values.T)
    return base_signals

  def custom_fitness_(self,y_true, y_pred, sample_weight):
    """
    Custom fitness function to evaluate chromosomes using the VectorBacktest.
    y_true, sample_weight are ignored since we're only using y_pred (chromosome signals).
    """
    try:
        #Ensure signals (chromosomes) are 1D
        if y_pred.ndim > 1:
            y_pred = y_pred.ravel()
        discrete_signals = y_pred
        #Check if discrete_signals is empty
        if discrete_signals.size == 0:
             return 0.0
        backtest = VectorBacktestGP(self.train_dataset, discrete_signals)
        sharpe_ratio = backtest.sharpe_capital()
        if (np.isnan(sharpe_ratio)or np.isinf(sharpe_ratio)):
            return 0.0
        return sharpe_ratio
    except Exception as e:
        print(f"Error in fitness evaluation: {e}")
        return -1e6
  def train_gplearn(self):
    self.est_gp.fit(self.X_train, self.y_train.ravel())

  def test_gplearn(self):
    final_generation = self.est_gp._programs[-1]
    test_scores = []

    for prog in final_generation:
      preds = prog.execute(self.X_test)
      if preds.size == 0:
          new_signals = np.zeros(len(self.test_dataset))
      else:
          new_signals = np.where(preds >self.signal_threshold, 1,
                            np.where(preds < -self.signal_threshold, -1, 0)).astype(int)

      unique_values, counts = np.unique(new_signals, return_counts=True)

      new_sharpe = VectorBacktestGP(self.test_dataset, new_signals).sharpe_capital()
      if np.isfinite(new_sharpe):
        test_scores.append((round(float(new_sharpe),3), prog))
    return final_generation,test_scores

  def perform_oos(self,fin_gen,oos_test_df):
    test_alpha_signals=self.dist_preprocess(oos_test_df,0,len(oos_test_df),num_ind)
    test_signals = pd.DataFrame(test_alpha_signals.T)
    X_test = np.nan_to_num(test_signals)
    test_dataset = oos_test_df[['Close']]
    test_scores = []
    gpportfolio_arr=[]
    for prog in fin_gen:
      preds = prog.execute(X_test)
      if preds.size == 0:
         new_signals = np.zeros(len(test_dataset))
      else:
        new_signals = np.where(preds >self.signal_threshold, 1,
                          np.where(preds < -self.signal_threshold, -1, 0)).astype(int)
        fin_obj=VectorBacktestGP(test_dataset, new_signals)

        fin_sharpe=fin_obj.sharpe_capital()
        fin_multi=fin_obj.return_mdd()
        if np.isfinite(fin_sharpe):
          test_score=(round(float(fin_sharpe),3), fin_multi[0], fin_multi[1])
          test_scores.append(test_score)
          gpportfolio_arr.append(fin_obj.portfolio)

    return test_scores,gpportfolio_arr


  def finalised_result(self):
    avg_res=0.0
    n_top = max(10, int(len(self.test_scores) * 0.01))
    top_test_strategies_ = self.test_scores[:n_top]
    print(f"Top {n_top} strategies in test set:")
    for s, p in top_test_strategies_:
      avg_res+=s
    print(f"Sharpe {(avg_res/n_top):.4f}")
    return avg_res/n_top

In [ ]:
compdf=pd.read_csv("zscored_indicators_spy.csv").drop(labels=['Unnamed: 0','Date'], axis=1, errors='ignore')
num_ind=53
compdf['fwd_return'] = compdf['Close'].pct_change().shift(1)
compdf=compdf.dropna()
final_gen=None
for sft_win in range(21):
  start_day,end_day,start_col=150*sft_win,1000+150*sft_win,6
  gplearn_obj=CompGPlearn(compdf,start_day,1000,500,0.01)
  if(sft_win==20):
    final_gen=gplearn_obj.final_generation
  print(f"{sft_win} timeperiod done")

## MLP Results


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
class VectorBacktestMLP:
  def __init__(self,dataset,terminal_signals):
    self.df = dataset.copy()
    self.test_signal = np.array(terminal_signals)
    assert self.test_signal.shape[1] == len(self.df), "Signals must match dataset length"
    shifted_signals = np.roll(self.test_signal, shift=1, axis=1)

    #Set the last position to -1 (exit) for all strategies
    shifted_signals[:, -1] = -1
    shifted_signals[:,0] = 0
    signal_df = pd.DataFrame(
            shifted_signals.T,
            index=self.df.index,
            columns=[f'sig{i+1}' for i in range(self.test_signal.shape[0])]
        )
    entries_df = signal_df == 1
    exits_df   = signal_df == -1
    asset_label ="SP500"
    entries = entries_df.vbt.stack_index(pd.Index([asset_label] * entries_df.shape[1], name="asset"))
    exits   = exits_df.vbt.stack_index(pd.Index([asset_label] * exits_df.shape[1], name="asset"))

    price_df = self.df[['Close']].copy()
    price_df.columns = [asset_label]
    price_df.columns.name = 'asset'

    self.portfolio = vbt.Portfolio.from_signals(
            price_df,
            entries,
            exits,
            init_cash=10000,
            slippage=0.0001,
            fees=0.0001,
            freq='1D'
        )
  def port_ret(self):
        return self.portfolio
  def fitness(self,type="sharpe"):
      metrics=self.port_ret()
      if(type=="sharpe"):
        return metrics.sharpe_ratio()
      elif(type=="max_drawdown"):
        return -metrics.max_drawdown()
      elif(type=="calmar_ratio"):
        return metrics.calmar_ratio()
      elif(type=="sortino_ratio"):
        return metrics.sortino_ratio()
      elif(type=="omega_ratio"):
        return metrics.omega_ratio()

In [ ]:
class MLPTraderBatch:
    def __init__(self, df, train_start=0,train_len=1000, test_len=500,
                 signal_threshold=0.01, hidden_dim=64,
                 lr=1e-3, epochs=30, n_strategies=1):
        """MLP-based signal generator with batch backtesting"""
        self.df = df.reset_index(drop=True)
        self.train_df = df.iloc[train_start:train_len].copy().reset_index(drop=True)
        self.test_df = df.iloc[train_start+train_len:train_start+train_len+test_len].copy().reset_index(drop=True)


        self.signal_threshold = signal_threshold
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.epochs = epochs
        self.n_strategies = n_strategies  #multiple MLP outputs in parallel
        self.X_train = torch.tensor(self.train_df.drop(columns=['Open', 'High', 'Low', 'Close','Volume','fwd_return']).values, dtype=torch.float32)
        self.y_train = torch.tensor(self.train_df['fwd_return'].values, dtype=torch.float32)
        self.X_test = torch.tensor(self.test_df.drop(columns=['Open', 'High', 'Low', 'Close','Volume','fwd_return']).values, dtype=torch.float32)
        self.y_test = torch.tensor(self.test_df['fwd_return'].values, dtype=torch.float32)
        self.model = nn.Sequential(
            nn.Linear(self.X_train.shape[1], hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_strategies)
        )
        print(self.model[0])  #should be Linear(in_features=53, out_features=64, bias=True)


        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)

    def _to_signals(self, preds: np.ndarray) -> np.ndarray:
        """
        Convert continuous predictions to discrete {-1, 0, 1} signals.
        preds: shape (n_samples, n_strategies)
        """
        return np.where(preds > self.signal_threshold, 1,
               np.where(preds < -self.signal_threshold, -1, 0)).T  #shape: (n_strategies, n_samples)

    def custom_loss(self, preds, y_true):
        if preds.dim() == 1:
            preds = preds.unsqueeze(1)
        y_true = y_true.unsqueeze(1).expand_as(preds)

        vx = preds - preds.mean(dim=0, keepdim=True)
        vy = y_true - y_true.mean(dim=0, keepdim=True)

        corr = (vx * vy).mean(dim=0) / (vx.std(dim=0) * vy.std(dim=0) + 1e-8)

        return -corr.mean()  #we want to maximize corr

    def train(self):
      for epoch in range(self.epochs):
          self.optimizer.zero_grad()
          outputs = self.model(self.X_train)
          loss = self.custom_loss(outputs, self.y_train)
          loss.backward()
          self.optimizer.step()
          if epoch % 5 == 0:
              print(f"Epoch {epoch}, Loss {loss.item():.4f}")
      with torch.no_grad():
        grads = self.model[0].weight.grad  # shape: (hidden_dim, num_ind)
        grad_per_feature = grads.abs().sum(dim=0).cpu().numpy()
        print("Gradient magnitude per feature:", grad_per_feature)
        print("Any unused features?", (grad_per_feature == 0).any())

    def evaluate(self):
        """Evaluate on test dataset using batch backtest.Returns per-strategy Sharpe and MDD."""
        with torch.no_grad():
            preds = self.model(self.X_test).cpu().numpy()
        discrete_signals = self._to_signals(preds)
        backtest = VectorBacktestMLP(self.test_df[['Close']], discrete_signals)

        sharpes = backtest.fitness("sharpe")
        mdds = backtest.fitness("max_drawdown")
        return sharpes, mdds, backtest.port_ret()
    def evaluate_oos(self, oos_df):
        X_oos = torch.tensor(
            oos_df.drop(columns=['Open','High','Low','Close','Volume','fwd_return']).values,
            dtype=torch.float32
        )
        with torch.no_grad():
            preds = self.model(X_oos).cpu().numpy()
        discrete_signals = self._to_signals(preds)
        backtest = VectorBacktestMLP(oos_df[['Close']], discrete_signals)
        sharpes = backtest.fitness("sharpe")
        mdds = backtest.fitness("max_drawdown")
        return sharpes, mdds, backtest.port_ret()

In [ ]:
compdf=pd.read_csv("zscored_indicators_spy.csv").drop(labels=['Unnamed: 0','Date'], axis=1, errors='ignore')
num_ind=53
compdf['fwd_return'] = compdf['Close'].pct_change().shift(1)
compdf=compdf.dropna()
tot_len=len(compdf)
def run_mlp_pipeline(st,is_last=False):
  mlp_trader = MLPTraderBatch(compdf, train_start=0,train_len=1000, test_len=500,
                              signal_threshold=0.01, hidden_dim=64,
                              lr=1e-3, epochs=30, n_strategies=10000)

  mlp_trader.train()
  sharpes, mdds, portfolio = mlp_trader.evaluate()
  sharpe_mdd_arr = []

  for i in range(len(sharpes)):
      if not np.isinf(sharpes.iloc[i]):
          sharpe_mdd_arr.append((round(sharpes.iloc[i], 3),
                                round(-mdds.iloc[i], 3),
                                i))
  sharpe_mdd_arr.sort(key=lambda x: (x[0], x[1]), reverse=True)

  #For top N strategies, add annualized return
  top_n = 10
  final_results = []
  for sharpe, mdd, idx in sharpe_mdd_arr[:top_n]:
      ann_ret = round(portfolio.iloc[idx].annualized_return(), 3)
      final_results.append((sharpe, ann_ret, mdd))

  print(final_results)
  if is_last:
    oos_df = compdf.iloc[st+1000:tot_len].copy().reset_index(drop=True)  # for example
    sharpes_oos, mdds_oos, portfolio_oos = mlp_trader.evaluate_oos(oos_df)

    oos_arr = []
    for i in range(len(sharpes_oos)):
        if not np.isinf(sharpes_oos.iloc[i]):
            oos_arr.append((round(sharpes_oos.iloc[i], 3),
                            round(-mdds_oos.iloc[i], 3),
                            i))
    oos_arr.sort(key=lambda x: (x[0], x[1]), reverse=True)

    oos_results = []
    for sharpe, mdd, idx in oos_arr[:top_n]:
        ann_ret = round(portfolio_oos.iloc[idx].annualized_return(), 3)
        oos_results.append((sharpe, ann_ret, mdd))

    print("OOS top strategies:", oos_results)
    best_ind=oos_arr[0][2]
    print("Best strategy index:", best_ind)
    print("Best strategy:",oos_arr[0])
    return portfolio_oos.iloc[best_ind]
  else:
    return portfolio

## PSO Results


In [ ]:
import warnings
from typing import List,Dict,Optional, Tuple
import numpy as np
import pandas as pd

class VectorBacktest1:
    def __init__(self, dataset, terminal_signals, fusion=False):
        """
        Perform backtesting on trading strategies using decision trees for signal generation.
        The backtest is conducted using the VectorBT library for portfolio simulation.

        Args:
            dataset (pd.DataFrame): Price data (expects 'Close' column).
            terminal_signals (np.ndarray): Discrete signals {-1, 0, 1}
        """
        self.df = dataset.copy()
        if terminal_signals.ndim == 1:
            terminal_signals = terminal_signals.reshape(1, -1)

        self.test_signal = np.array(terminal_signals)

        if self.test_signal.shape[1] != len(self.df):
             min_len = min(self.test_signal.shape[1], len(self.df))
             self.test_signal = self.test_signal[:, :min_len]
             self.df = self.df.iloc[:min_len]
             warnings.warn(f"Signal length ({terminal_signals.shape[1]}) != dataset length ({len(dataset)}). Truncating to {min_len}.")
             if min_len == 0:
                  raise ValueError("Signal or dataset has zero length after alignment.")
        shifted_signals = np.roll(self.test_signal, shift=1, axis=1)
        shifted_signals[:, 0] = 0
        shifted_signals[:, -1] = -1

        signal_df = pd.DataFrame(
            shifted_signals.T,
            index=self.df.index,
            columns=[f'sig{i+1}' for i in range(self.test_signal.shape[0])]
        )

        entries_df = signal_df == 1
        exits_df = signal_df == -1
        asset_label = "asset"
        if 'asset' in dataset.columns.names:
             asset_label = dataset.columns.get_level_values('asset')[0]
        elif len(dataset.columns) == 1:
             asset_label = dataset.columns[0]
        price_df = self.df[['Close']].copy()
        price_df.columns = pd.Index([asset_label], name='asset')


        entries = entries_df.vbt.stack_index(pd.Index([asset_label] * entries_df.shape[1], name="asset"))
        exits = exits_df.vbt.stack_index(pd.Index([asset_label] * exits_df.shape[1], name="asset"))
        self.portfolio = vbt.Portfolio.from_signals(
            price_df,
            entries,
            exits,
            init_cash=10000,
            slippage=0.0001,
            fees=0.0001,
            freq='1D'
        )

    def port_ret(self):
        return self.portfolio

    def fitness(self, type="sharpe"):
        metrics=self.port_ret()
        if(type=="sharpe"):
          return metrics.sharpe_ratio()
        elif(type=="max_drawdown"):
          return -metrics.max_drawdown()
        elif(type=="calmar_ratio"):
          return metrics.calmar_ratio()
        elif(type=="sortino_ratio"):
          return metrics.sortino_ratio()
        elif(type=="omega_ratio"):
          return metrics.omega_ratio()
        else:
            warnings.warn(f"Unknown fitness type: {type}")
            return -np.inf

def _to_signals(preds: np.ndarray, signal_threshold: float) -> np.ndarray:
    """ Convert continuous predictions to discrete {-1, 0, 1} signals. """
    preds=np.asarray(preds)
    if preds.ndim == 1 or preds.ndim==2:
        signals = np.where(preds > signal_threshold, 1,
                   np.where(preds < -signal_threshold, -1, 0))
        return signals
    else:
        raise ValueError("preds must be 1D or 2D array")

def evaluate_combination(weights: np.ndarray,
                         train_df: pd.DataFrame,
                         base_alpha_signals: np.ndarray,
                         signal_threshold: float):
    """Evaluates weights by combining a 2D base_alpha_signals array and running a backtest."""

    try:
        #base_alpha_signals_window shape: (num_alphas, num_samples)
        weights = np.asarray(weights).ravel()
        base_alpha_signals = np.asarray(base_alpha_signals)
        if base_alpha_signals.ndim != 2:
            raise ValueError("base_alpha_signals must be 2D array (num_alphas, num_samples)")
        signal_len,num_alphas = base_alpha_signals.shape
        if len(weights) != num_alphas:
            raise ValueError(f"Weight vector dim ({len(weights)}) != num alphas ({num_alphas})")

        #Normalize weights
        norm_weights = weights / (np.sum(np.abs(weights)) + 1e-9)

        #Combine weighted signals across alphas
        combined_signal_continuous = np.dot(base_alpha_signals,norm_weights)

        #Discretize the combined signal
        discrete_signal = _to_signals(combined_signal_continuous, signal_threshold)
        if len(discrete_signal) != len(train_df):
            raise ValueError(f"Discrete signal length {len(discrete_signal)} != train_df length {len(train_df)}")
        return discrete_signal.flatten()

    except Exception as e:
        print(f"Signal evaluation error: {e}")
        return np.zeros(base_alpha_signals.shape[1])

In [ ]:
#Particle Class
class Particle:
    def __init__(self, dim: int, bounds: list):
        self.position = np.array([np.random.uniform(b[0], b[1]) for b in bounds])
        vel_val=abs(bounds[0][1]-bounds[0][0])*0.1
        self.velocity = np.random.uniform(-vel_val,vel_val, dim)
        self.fitness = -np.inf
        self.pbest_position = self.position.copy()
        self.pbest_fitness = -np.inf
        self.bounds = bounds
    def update_velocity(self, gbest_position: np.ndarray, w: float, c1: float, c2: float):
        r1 = np.random.rand(len(self.position))
        r2 = np.random.rand(len(self.position))
        cognitive_velocity = c1 * r1 * (self.pbest_position - self.position)
        social_velocity = c2 * r2 * (gbest_position - self.position)
        self.velocity = w * self.velocity + cognitive_velocity + social_velocity
        max_vel = np.array([abs(b[1]-b[0])*0.5 for b in self.bounds])
        self.velocity = np.clip(self.velocity, -max_vel, max_vel)
    def update_position(self):
        self.position += self.velocity
        for i in range(len(self.position)):
            self.position[i] = np.clip(self.position[i], self.bounds[i][0], self.bounds[i][1])

#PSO Optimizer Function
def pso_optimizer(
    train_df: pd.DataFrame,
    base_alpha_signals_window:np.ndarray,
    signal_threshold: float,
    dim: int,
    bounds: list,
    num_particles: int = 10000,
    max_iterations: int = 40,
    w: float = 0.5, c1: float = 1.5, c2: float = 1.5,
    initial_swarm_positions= None
) -> tuple:

    swarm = [Particle(dim, bounds) for _ in range(num_particles)]
    if initial_swarm_positions is not None and len(initial_swarm_positions) > 0:
        num_seed = min(len(swarm), len(initial_swarm_positions))
        for i in range(num_seed):
            noise = np.random.normal(0, np.mean([abs(b[1]-b[0]) for b in bounds]) * 0.05, dim)
            swarm[i].position = np.clip(initial_swarm_positions[i] + noise,
                                        [b[0] for b in bounds], [b[1] for b in bounds])
            swarm[i].pbest_position = swarm[i].position.copy()

    gbest_position = None
    gbest_fitness = -np.inf
    for iteration in range(max_iterations):
        #Generate signals for all particles
        all_signals = []
        for particle in swarm:
            signal = evaluate_combination(particle.position,train_df,base_alpha_signals_window, signal_threshold)
            all_signals.append(signal)
        all_signals = np.vstack(all_signals)  #shape: (num_particles, n_samples)

        #Run backtest across all signals
        backtest = VectorBacktest1(train_df[['Close']], all_signals)
        fitness_values = backtest.fitness()
        #Update particles
        for i, particle in enumerate(swarm):
            particle.fitness = fitness_values.iloc[i] if np.isfinite(fitness_values.iloc[i]) else -np.inf

            if particle.fitness > particle.pbest_fitness:
                particle.pbest_fitness = particle.fitness
                particle.pbest_position = particle.position.copy()

            if particle.fitness > gbest_fitness:
                gbest_fitness = particle.fitness
                gbest_position = particle.position.copy()

            particle.update_velocity(gbest_position, w, c1, c2)
            particle.update_position()
        print(f"Iter {iteration+1}/{max_iterations} | gbest_fitness (Train Sharpe): {gbest_fitness:.4f}")

    sorted_particles = sorted(swarm, key=lambda p: p.fitness, reverse=True)
    top_particles = sorted_particles[:10]
    top_positions = np.array([p.position for p in top_particles])
    top_fitness = np.array([p.fitness for p in top_particles])
    return gbest_position, gbest_fitness, top_positions, top_fitness


#Helper function to safely extract scalar values
def extract_scalar(value):
    """Safely extract scalar from pandas Series or return the value itself."""
    if isinstance(value, pd.Series):
        return float(value.iloc[0])
    return float(value)

In [ ]:
#PSO pipeline function
def run_pso_pipeline(full_df: pd.DataFrame,
                     all_base_signals: np.ndarray,
                     train_start: int, train_len: int, test_len: int,
                     signal_threshold: float,
                     weight_bounds: list,
                     num_alphas: int,
                     pso_particles: int,
                     pso_iterations: int,
                     is_last: bool = False,
                     prev_best_weights: Optional[np.ndarray] = None
                    ) -> Optional[tuple]:
    train_end = train_start + train_len
    test_end = train_end + test_len
    if train_end > len(full_df):
        print(f"Skipping window starting at {train_start}: Not enough data for training.")
        return None
    if test_end > len(full_df) and not is_last:
        print(f"Skipping window starting at {train_start}: Not enough data for testing.")
        return None

    train_df = full_df.iloc[train_start:train_end].copy().reset_index(drop=True)
    actual_test_end = min(test_end, len(full_df))
    test_df = full_df.iloc[train_end:actual_test_end].copy().reset_index(drop=True)
    train_alpha_signals = all_base_signals[train_start:train_end, :]
    print(f"Optimizing PSO for Window (Train Indices {train_start}-{train_end})")

    best_weights_train, best_fitness_train, top_positions, top_fitness = pso_optimizer(
        train_df=train_df,
        base_alpha_signals_window=train_alpha_signals,
        signal_threshold=signal_threshold,
        dim=num_alphas,
        bounds=weight_bounds,
        num_particles=pso_particles,
        max_iterations=pso_iterations,
        initial_swarm_positions=[prev_best_weights] if prev_best_weights is not None else None
    )
    print(f"Training Complete. Best Train Sharpe: {best_fitness_train:.4f}")
    print(f"Evaluating Best Weights on Test Window (Indices {train_end}-{actual_test_end})")

    oos_results = []
    port_ret_best = None
    if len(test_df) > 0:
        test_alpha_signals = all_base_signals[train_end:actual_test_end, :]
        all_weight_sets = np.vstack([best_weights_train[None, :], top_positions])
        all_signals_test = []
        for weights in all_weight_sets:
            norm_weights = weights / (np.sum(np.abs(weights)) + 1e-9)
            combined_cont = np.dot(test_alpha_signals, norm_weights)
            discrete_signal = _to_signals(combined_cont.reshape(1, -1), signal_threshold)
            all_signals_test.append(discrete_signal.flatten())

        all_signals_test = np.vstack(all_signals_test)
        test_backtest = VectorBacktest1(test_df[['Close']], all_signals_test)
        fitnesses = test_backtest.fitness()

        sharpe_test = fitnesses
        mdd_test = test_backtest.fitness("max_drawdown")

        oos_arr = []
        for i in range(len(sharpe_test)):
            if not np.isinf(sharpe_test.iloc[i]):
                oos_arr.append((round(sharpe_test.iloc[i], 3),
                                round(-mdd_test.iloc[i], 3), i))

        if len(oos_arr):
            oos_arr.sort(key=lambda x: (x[0], x[1]), reverse=True)
            oos_results = []
            for sharpe, mdd, idx in oos_arr[:10]:
                ann_ret = round(test_backtest.port_ret().iloc[idx].annualized_return(), 3)
                oos_results.append((sharpe, ann_ret, mdd))

            print("Window OOS top strategies:", oos_results)
        else:
            print("Test set is empty, skipping evaluation.")
    else:
        print("Test set is empty, skipping evaluation.")

    if is_last:
        oos_start = train_end
        oos_end=len(full_df)-465
        print(f"The start and end are {oos_start},{oos_end}")
        oos_df = full_df.iloc[oos_start:oos_end].copy().reset_index(drop=True)
        oos_alpha_signals = all_base_signals[oos_start:, :]

        all_weight_sets = np.vstack([best_weights_train[None, :], top_positions])
        all_signals_oos = []
        for weights in all_weight_sets:
            norm_weights = weights / (np.sum(np.abs(weights)) + 1e-9)
            combined_cont = np.dot(oos_alpha_signals, norm_weights)
            discrete_signal = _to_signals(combined_cont.reshape(1, -1), signal_threshold)
            all_signals_oos.append(discrete_signal.flatten())

        all_signals_oos = np.vstack(all_signals_oos)
        oos_backtest = VectorBacktest1(oos_df[['Close']], all_signals_oos)
        fitnesses = oos_backtest.fitness()

        sharpe_test = fitnesses
        mdd_test = oos_backtest.fitness("max_drawdown")

        oos_arr = []
        for i in range(len(sharpe_test)):
            if not np.isinf(sharpe_test.iloc[i]):
                oos_arr.append((round(sharpe_test.iloc[i], 3),
                                round(-mdd_test.iloc[i], 3), i))

        if len(oos_arr):
            oos_arr.sort(key=lambda x: (x[0], x[1]), reverse=True)

            oos_results = []
            best_idx = oos_arr[0][2]
            port_ret_best = oos_backtest.port_ret().iloc[best_idx]

            for sharpe, mdd, idx in oos_arr[:10]:
                ann_ret = round(oos_backtest.port_ret().iloc[idx].annualized_return(), 3)
                oos_results.append((sharpe, ann_ret, mdd))

            print("OOS top strategies:", oos_results)
        else:
            print("OOS Test set is empty, skipping evaluation.")
    else:
        print("OOS Test set is empty, skipping evaluation.+++++++++++++++++")
    if is_last:
        return best_weights_train, oos_results, port_ret_best
    else:
        return best_weights_train, oos_results

In [ ]:
compdf=pd.read_csv("zscored_indicators_spy.csv").drop(labels=['Unnamed: 0','Date'], axis=1, errors='ignore')
alpha_cols = [col for col in compdf.columns if col.startswith('alpha')]
if not alpha_cols:
    alpha_cols = [col for col in compdf.columns
                  if col not in ['Open', 'High', 'Low', 'Close', 'Volume', 'fwd_return']]
    print(f"Warning: No columns starting with 'alpha'. Using {len(alpha_cols)} other columns as signals.")

num_alphas = len(alpha_cols)
print(f"Using {num_alphas} alpha signals: {alpha_cols[:5]}..." if len(alpha_cols) > 5 else f"Using {num_alphas} alpha signals")
all_base_signals=compdf[alpha_cols].to_numpy()
full_data_df = compdf[['Close']].copy()
print(f"Total data points: {len(full_data_df)}")


TRAIN_LEN = 1000
TEST_LEN = 500
STEP_SIZE = 150  # Shift size for walk-forward
SIGNAL_THRESHOLD = 0.01
WEIGHT_BOUNDS = [(-1.0, 1.0)] * num_alphas
PSO_PARTICLES = 10000
PSO_ITERATIONS = 30
results_list = []
best_weights_previous = None

for i in range(21):
    start = i * STEP_SIZE
    end=start+TRAIN_LEN
    is_last_run = (i==20)

    print(f"\n{'='*60}")
    print(f"WINDOW {i}: Start Index = {start}")
    print(f"Train: {start} to {start + TRAIN_LEN}")
    print(f"Test: {start + TRAIN_LEN} to {start+TRAIN_LEN+TEST_LEN}")
    if is_last_run:
        print("Last window")
    print(f"{'='*60}")

    best_weights_current,oos_res,best_portfolio = run_pso_pipeline(
        full_df=full_data_df,
        all_base_signals=all_base_signals,
        train_start=start,
        train_len=TRAIN_LEN,
        test_len=TEST_LEN,
        signal_threshold=SIGNAL_THRESHOLD,
        weight_bounds=WEIGHT_BOUNDS,
        num_alphas=num_alphas,
        pso_particles=PSO_PARTICLES,
        pso_iterations=PSO_ITERATIONS,
        is_last=is_last_run,
        prev_best_weights=best_weights_previous
    )

    if best_weights_current is not None:
        best_weights_previous = best_weights_current
        results_list.append({
            'window': i,
            'start_idx': start,
            'weights': best_weights_current.copy(),
            'window_result': oos_res
        })

        print(f"Window {i} Summary")
        print(f"{results_list[-1]['window_result']}, {len(results_list[-1]['window_result'])}")
    else:
        print(f"Window {i} (starting at {start}) failed or skipped.")

print(f"{'='*60}")
print("EXECUTION COMPLETE")
print(f"{'='*60}")